# NYC Housing x Transportation: Data Story (Milestone 1)
**Course:** Data Engineering and Visualization, Winter 2026 | **Deadline:** 9 Nov 2026

**How to work in this notebook (read first)**
- Each member works ONLY inside their own section (marked with their name) to avoid edit clashes.
- Do not rename the shared variables: `df_housing_raw`, `df_transport_raw`, `df_housing`, `df_transport`, `merged`.
- Every decision (drop vs impute, outlier rule, join level) needs a markdown cell explaining **why**.
- Log every AI prompt you use in section 9.

| Member | Section |
|---|---|
| M1 Housing lead | 2, 3, 7 (RQ1-2) |
| M2 Transport lead | 2, 4, 7 (RQ3-4) |
| M3 Integration lead | 0, 1, 5, 6, 7 (RQ5-6) |
| M4 Website + story lead | 7 (RQ7-8), 8, 10 |

## 0. Setup and dataset links (M3)
- Housing: https://data.cityofnewyork.us/Housing-Development/DOB-NOW-Build-Job-Application-Filings/w9ak-ipjd
- Crashes: https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95

Run the next two cells first. The download takes a few minutes. It retries dropped connections and saves pages to `soda_cache/`, so if it fails just re-run the cell and it resumes. Everyone runs it once in their own Colab copy.

In [ ]:
import re, os, time, hashlib
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from urllib.parse import urlencode

# Both IDs verified on data.cityofnewyork.us (Oct 2026):
HOUSING_ID   = 'w9ak-ipjd'   # DOB NOW: Build - Job Application Filings (~969K rows, 95 columns)
TRANSPORT_ID = 'h9gi-nx95'   # Motor Vehicle Collisions - Crashes (~2.27M rows, 29 columns)
START_DATE   = '2022-01-01'  # sampling window start (justified in section 1)
API = 'https://data.cityofnewyork.us/resource/{}.csv'

def find_col(cols, candidates, regex=None, required=True):
    """Return the first matching column name. Column names differ between the CSV
    download and the API, so we look them up instead of hard-coding them."""
    cols = list(cols)
    for c in candidates:
        if c in cols:
            return c
    if regex:
        hits = [c for c in cols if re.search(regex, c, flags=re.I)]
        if hits:
            return hits[0]
    if required:
        raise KeyError(f'None of {candidates} (regex={regex}) found. Available columns: {cols}')
    return None

def soda_get(dataset_id, params, retries=6, wait=4):
    """Download one page. Retries on dropped connections ('Connection reset by peer', timeouts)."""
    url = API.format(dataset_id) + '?' + urlencode(params)
    for attempt in range(1, retries + 1):
        try:
            return pd.read_csv(url, low_memory=False)
        except Exception as e:
            if attempt == retries:
                raise
            print(f'    {type(e).__name__}: retry {attempt}/{retries - 1} in {wait * attempt}s')
            time.sleep(wait * attempt)

def probe_columns(dataset_id):
    """Download 5 rows just to learn the real column names."""
    return list(soda_get(dataset_id, {'$limit': 5}).columns)

CACHE_DIR = 'soda_cache'          # pages are saved here, so a re-run resumes instead of starting over
os.makedirs(CACHE_DIR, exist_ok=True)

def load_since(dataset_id, date_col, select=None, start=START_DATE, page=50_000, max_rows=3_000_000):
    """Download rows with date_col >= start in pages of 50k (stable order by :id).
    - select: list of columns to download. Slimming 95 columns to ~15 makes the download much faster and lighter.
    - each page is cached on disk; a dropped connection only repeats that one page."""
    pages, offset = [], 0
    while offset < max_rows:
        params = {'$where': f"{date_col} >= '{start}T00:00:00'", '$order': ':id', '$limit': page, '$offset': offset}
        if select:
            params['$select'] = ','.join(select)
        key = hashlib.md5(urlencode(params).encode()).hexdigest()[:12]
        path = os.path.join(CACHE_DIR, f'{dataset_id}_{key}.pkl')
        if os.path.exists(path):
            chunk = pd.read_pickle(path)
        else:
            chunk = soda_get(dataset_id, params)
            chunk.to_pickle(path)
        pages.append(chunk)
        print(f'  {dataset_id}: {offset + len(chunk):,} rows')
        if len(chunk) < page:
            break
        offset += page
    return pd.concat(pages, ignore_index=True)

housing_cols   = probe_columns(HOUSING_ID)
transport_cols = probe_columns(TRANSPORT_ID)
print('HOUSING columns  :', housing_cols)       # <- everyone: read this once
print('TRANSPORT columns:', transport_cols)

# --- which columns to download (slim = faster, less memory, fewer connection drops) ---
HOUSING_DATE_COL = find_col(housing_cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
HOUSING_EXTRA_COLS = []     # M1: add any other housing columns you want for EDA (names from the list printed above)
housing_core = ['job_filing_number', 'filing_status', 'house_no', 'street_name', 'borough', 'block', 'lot', 'bin', 'commmunity_board']
housing_pattern = r'^(zip|latitude|longitude|gis_lat|gis_lon|job_type|filing_date|current_status_date|initial_cost|proposed_dwelling|existing_dwelling|proposed_no_of_stories|building_type|work_type|council|census|nta)'
housing_select = [c for c in housing_cols if c in housing_core or c in HOUSING_EXTRA_COLS or re.search(housing_pattern, c, flags=re.I)]
for c in [HOUSING_DATE_COL, find_col(housing_cols, ['job_type'], regex=r'job.*type'),
          find_col(housing_cols, ['zip_code', 'zip', 'zipcode'], regex=r'^zip', required=False),
          find_col(housing_cols, ['latitude', 'gis_latitude'], regex=r'^(gis_)?lat', required=False),
          find_col(housing_cols, ['longitude', 'gis_longitude'], regex=r'^(gis_)?lon', required=False)]:
    if c and c not in housing_select:
        housing_select.append(c)

transport_wanted = ['collision_id', 'crash_date', 'crash_time', 'borough', 'zip_code', 'latitude', 'longitude', 'on_street_name',
                    'number_of_persons_injured', 'number_of_persons_killed', 'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
                    'number_of_cyclist_injured', 'number_of_cyclist_killed', 'number_of_motorist_injured', 'number_of_motorist_killed']
transport_select = [c for c in transport_wanted if c in transport_cols]
print('\nDownloading', len(housing_select), 'housing columns:', housing_select)

df_housing_raw   = load_since(HOUSING_ID, HOUSING_DATE_COL, select=housing_select)
df_transport_raw = load_since(TRANSPORT_ID, 'crash_date', select=transport_select)
print(df_housing_raw.shape, df_transport_raw.shape)

In [ ]:
import os
files = os.listdir('soda_cache') if os.path.exists('soda_cache') else []
print(len(files), 'cached pages')
print(sorted(files)[:20])

## 1. Dataset overview and selection reasoning (M3 writes; M1 and M2 add details for their dataset)

| | Housing | Transportation |
|---|---|---|
| Dataset | DOB NOW: Build - Job Application Filings (`w9ak-ipjd`, catalog ID **H5**) | Motor Vehicle Collisions - Crashes (`h9gi-nx95`, catalog ID **T4**) |
| Size (NYC Open Data page, Oct 2026) | ~969K rows, 95 columns | ~2.27M rows, 29 columns |
| Rows and columns used (2022 onward) | 734,423 rows, 23 columns downloaded; 370,870 rows after cleaning | 448,091 rows, 16 columns downloaded; 420,067 rows after cleaning |
| Geographic attributes | borough, street name, block, lot, BIN, community board, latitude/longitude (the published ZIP is unreliable, so the ZIP is derived from the coordinates) | borough, zip_code, latitude, longitude, on/off/cross street |
| Temporal attributes | `filing_date` | crash_date, crash_time |
| Both have >= 9 columns | yes | yes |

**Why this pair:** both are tied to the same places and months, so they can be compared at ZIP x month without matching individual rows. Question: *do areas with heavy construction activity also see more street-safety problems?*

**Sampling (required justification):** both tables are large, so we download rows from **2022-01-01** onward through the SoQL API in 50k-row pages. 2022 onward gives ~4 years (enough for seasonality and lags), keeps the data recent, and fits in Colab memory. The data were downloaded from the NYC Open Data API on 8 Nov 2026; the source is updated daily, so a later download may give slightly different counts.

**Known issues to document**
- Crash table: ZIP and borough are often blank (recovered from coordinates where possible, see section 4), coordinates of 0,0 exist, and the table is preliminary data.
- **Crash feed:** the crash data in our download runs through 3 November 2026 with no visible gap (about 6,900 to 7,700 crashes per month from July to October 2026). Our pipeline uses the common date window of both datasets and drops an unfinished last month as a safeguard.
- Housing table: column names differ slightly from the CSV download vs. the API (e.g. the community board column is spelled `commmunity_board`), so we look columns up with `find_col` instead of hard-coding them.
- **Housing table (ZIP and follow-up filings):** the published `zip` column does not describe the building's location (only 3.5% of filings had a ZIP equal to the nearest crash point's ZIP, see section 3), so the housing ZIP is derived from latitude/longitude. Only 51.3% of the filings are initial applications; the rest are follow-up filings on the same job, which would count one project several times, so the cleaned housing table keeps initial applications only. After cleaning, 370,870 of 734,423 downloaded rows remain (98.4% of the initial applications). `proposed_no_of_stories` is 47.5% empty and was dropped; dwelling-unit columns are empty by design for some job types.

**Integration plan:** aggregate both to **ZIP x year-month**, then join on `zip` + `year_month` (details and alternatives in section 5).

## 2. Shared EDA helpers (M1 + M2)

In [ ]:
def overview(df, name):
    print(f'=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols ===')
    display(pd.DataFrame({'dtype': df.dtypes, 'nulls': df.isna().sum(),
                          'null_%': (df.isna().mean()*100).round(1), 'unique': df.nunique()}))

def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - k*i, q3 + k*i

### 2b. Shared cleaning helpers (M3 wrote these; **M1 and M2 should reuse them**)
`clean_zip`, `clean_borough`, `clean_coords`, `add_time_cols`, `fill_zip_nearest`. Using the same functions on both datasets guarantees the join keys match exactly (5-digit string ZIPs, upper-case borough names, NYC-box coordinates).

In [ ]:
from sklearn.neighbors import BallTree

BOROUGHS = ['BRONX', 'BROOKLYN', 'MANHATTAN', 'QUEENS', 'STATEN ISLAND']
BORO_MAP = {'1': 'MANHATTAN', '2': 'BRONX', '3': 'BROOKLYN', '4': 'QUEENS', '5': 'STATEN ISLAND',
            'MN': 'MANHATTAN', 'BX': 'BRONX', 'BK': 'BROOKLYN', 'QN': 'QUEENS', 'SI': 'STATEN ISLAND'}
NYC_LAT, NYC_LON = (40.45, 40.95), (-74.30, -73.65)      # rough NYC bounding box (domain rule)

def clean_zip(s):
    """Keep the first 5 digits ('11201-1234' -> '11201'); anything outside NYC ZIP range -> NaN."""
    z = s.astype(str).str.extract(r'(\d{5})')[0]
    return z.where(pd.to_numeric(z, errors='coerce').between(10001, 11697))

def clean_borough(s):
    """Upper-case, strip, map numeric codes (1-5) and abbreviations; unknown -> NaN."""
    b = s.astype(str).str.strip().str.upper().str.replace(r'\.0$', '', regex=True).replace(BORO_MAP)
    return b.where(b.isin(BOROUGHS))

def clean_coords(lat, lon):
    """Coordinates outside the NYC box (including the 0,0 placeholder) are invalid -> NaN."""
    lat, lon = pd.to_numeric(lat, errors='coerce'), pd.to_numeric(lon, errors='coerce')
    ok = lat.between(*NYC_LAT) & lon.between(*NYC_LON)
    return lat.where(ok), lon.where(ok)

def add_time_cols(df, date_col):
    out = df.copy()
    out['date'] = pd.to_datetime(out[date_col], errors='coerce')
    out['year'], out['month'] = out['date'].dt.year, out['date'].dt.month
    out['year_month'] = out['date'].dt.strftime('%Y-%m')
    return out

def fill_zip_nearest(target, ref, max_km=0.5):
    """Recover a missing ZIP from the NEAREST reference point that has a ZIP (haversine BallTree).
    Only used if the nearest point is within max_km; otherwise the ZIP stays missing."""
    out = target.copy()
    out['zip'] = out['zip'].astype(object)                  # may be all-NaN float if the table had no ZIP column
    need = out['zip'].isna() & out['lat'].notna() & out['lon'].notna()
    r = ref.dropna(subset=['zip', 'lat', 'lon'])
    if need.sum() == 0 or len(r) == 0:
        return out
    tree = BallTree(np.radians(r[['lat', 'lon']].to_numpy()), metric='haversine')
    dist, idx = tree.query(np.radians(out.loc[need, ['lat', 'lon']].to_numpy()), k=1)
    km = dist[:, 0] * 6371.0088
    out.loc[need, 'zip'] = np.where(km <= max_km, r['zip'].to_numpy()[idx[:, 0]], None)
    return out

### Cleaning contract (read before cleaning)
Your cleaned table must end up in **`df_housing`** (M1) / **`df_transport`** (M2) with exactly these columns, so integration never breaks:

| Both | Housing only (M1) | Transport only (M2) |
|---|---|---|
| `date` (datetime), `year`, `month`, `year_month` ('2024-03'), `zip` (5-char string), `borough` (upper-case), `lat`, `lon` | `job_type` | `number_of_persons_injured`, `number_of_persons_killed`, `number_of_pedestrians_injured/killed`, `number_of_cyclist_injured/killed`, `number_of_motorist_injured/killed` (numeric, no nulls) |

`quick_clean_housing` / `quick_clean_transport` in section 5 show the minimum version. Do the real cleaning (with EDA, outlier handling and markdown justifications) and **overwrite** `df_housing` / `df_transport`. Do not change the column names.

## 3. HOUSING: EDA and pre-integration cleaning (**M1**)

In [ ]:
overview(df_housing_raw, 'Housing raw')
df_housing_raw.describe(include='all').T.head(30)

In [ ]:
print(df_housing_raw.shape)
print(list(df_housing_raw.columns))
print()
print(df_housing_raw['job_type'].value_counts(dropna=False))
print()
print(df_housing_raw.isna().mean().mul(100).round(1).sort_values(ascending=False).head(15))

In [ ]:
# 3a. Structure, duplicates, dates, borough and ZIP formats
print(df_housing_raw.dtypes)
print()
print('Duplicate job_filing_number:', df_housing_raw['job_filing_number'].duplicated().sum())
print('Fully duplicated rows     :', df_housing_raw.duplicated().sum())
print()
dates = pd.to_datetime(df_housing_raw['filing_date'], errors='coerce')
print('Unparseable dates:', dates.isna().sum())
print('Date range       :', dates.min(), '->', dates.max())
print()
print(df_housing_raw['borough'].value_counts(dropna=False))
print()
print('ZIP string lengths:', df_housing_raw['zip'].astype(str).str.len().value_counts().to_dict())
print('ZIP sample        :', df_housing_raw['zip'].astype(str).sample(10, random_state=1).tolist())
print()
nums = df_housing_raw[['initial_cost', 'existing_dwelling_units', 'proposed_dwelling_units']].apply(pd.to_numeric, errors='coerce')
print(nums.describe(percentiles=[.5, .9, .99, .999]).round(1))

In [ ]:
# 3b. EDA plots
dates = pd.to_datetime(df_housing_raw['filing_date'], errors='coerce')
fig, ax = plt.subplots(2, 2, figsize=(15, 10))

df_housing_raw['job_type'].value_counts().sort_values().plot.barh(ax=ax[0, 0], title='Filings by job type')

df_housing_raw['borough'].astype(str).str.upper().value_counts().plot.bar(ax=ax[0, 1], title='Filings by borough', rot=30)

monthly = dates.dt.to_period('M').value_counts().sort_index()
monthly.index = monthly.index.to_timestamp()
monthly = monthly[monthly.index < '2026-11-01']   # November 2026 is a partial month
monthly.plot(ax=ax[1, 0], title='Filings per month (complete months only)')

miss = df_housing_raw.isna().mean().mul(100).sort_values(ascending=False)
miss[miss > 0].plot.barh(ax=ax[1, 1], title='Missing values (%)')
ax[1, 1].invert_yaxis()

plt.tight_layout()
plt.show()

**EDA findings (M1).** The raw table has 734,423 filings from 1 Jan 2022 to 7 Nov 2026 and 23 columns. Alteration filings dominate (about 81%), followed by New Building (7.6%), Alteration CO (6.8%) and ALT-CO New Building with Existing Elements to Remain (2.2%); No Work and Full Demolition together are under 3%. Manhattan has the most filings (about 278K), then Brooklyn (202K), Queens (145K), the Bronx (73K) and Staten Island (37K). Monthly filings dip around the turn of each year and spike in early 2026, which we flag but cannot explain from this data alone (the unfinished month of November 2026 is left out of the plot). Missing values are concentrated in `proposed_no_of_stories` (47.5%), `existing_dwelling_units` (15.4%) and `proposed_dwelling_units` (8.5%); all other columns are under 2%. Issues found: 149 repeated filing numbers (3 of them exact duplicate rows); borough names in title case instead of the upper case the join key needs; ZIP values in mixed formats (728,574 five-character values, 5,060 ZIP+4 values, 289 malformed values and 500 missing); placeholder values (a cost of $800,010,000 and 111,111,111 existing dwelling units in one filing); and a `zip` column that does not describe the building's location (see the ZIP check below). The most important finding is that each job has exactly one initial application (filing number ending in -I) and later filings on the same job carry other suffixes. Only 51.3% of the distinct filings are initial applications, so counting every row would count the same project several times.

### ZIP check (M1)

Before cleaning, we tested whether the housing `zip` column can serve as the join key. For each filing with valid coordinates we compared its ZIP with the ZIP of the nearest crash record (within 0.5 km). Only 77.8% of the published ZIPs were valid NYC ZIPs, and only 3.5% matched the nearest crash ZIP, while the crash ZIP agreed with the filing's borough 99.9% of the time. Examples: Bronx and Queens filings carry Manhattan ZIP 10018, and some carry ZIPs outside NYC such as 07731. The column therefore does not describe the building's location, so it was not used. The ZIP is derived from coordinates in the cleaning step below.

In [ ]:
# DIAGNOSTIC: can we trust the housing 'zip' column?
ZIP_PREFIX_M1 = {'100': 'MANHATTAN', '101': 'MANHATTAN', '102': 'MANHATTAN', '103': 'STATEN ISLAND',
                 '104': 'BRONX', '110': 'QUEENS', '111': 'QUEENS', '112': 'BROOKLYN',
                 '113': 'QUEENS', '114': 'QUEENS', '116': 'QUEENS'}

ref = pd.DataFrame({'zip': clean_zip(df_transport_raw['zip_code'].astype(str)),
                    'lat': pd.to_numeric(df_transport_raw['latitude'], errors='coerce'),
                    'lon': pd.to_numeric(df_transport_raw['longitude'], errors='coerce')})
ref['lat'], ref['lon'] = clean_coords(ref['lat'], ref['lon'])
ref = ref.dropna()

d = df_housing_raw[['zip', 'borough', 'latitude', 'longitude']].copy()
d['own_zip'] = clean_zip(d['zip'].astype(str).str.strip())
d['borough'] = clean_borough(d['borough'])
d['lat'], d['lon'] = clean_coords(d['latitude'], d['longitude'])
d = d.dropna(subset=['lat', 'lon', 'borough'])

tree = BallTree(np.radians(ref[['lat', 'lon']].to_numpy()), metric='haversine')
dist, idx = tree.query(np.radians(d[['lat', 'lon']].to_numpy()), k=1)
d['crash_zip'] = np.where(dist[:, 0] * 6371.0088 <= 0.5, ref['zip'].to_numpy()[idx[:, 0]], None)

both = d['own_zip'].notna() & d['crash_zip'].notna()
d['own_matches_crash'] = d['own_zip'] == d['crash_zip']
d['crash_zip_matches_boro'] = d['crash_zip'].str[:3].map(ZIP_PREFIX_M1) == d['borough']

print(f"Rows with coordinates: {len(d):,}")
print(f"Housing ZIP is NYC-valid: {d['own_zip'].notna().mean():.1%}")
print(f"Where both exist, housing ZIP equals nearest-crash ZIP: {d.loc[both, 'own_matches_crash'].mean():.1%}")
print(f"Nearest-crash ZIP agrees with the borough label (sanity check of the reference): "
      f"{d.loc[d['crash_zip'].notna(), 'crash_zip_matches_boro'].mean():.1%}")
print()
print(d.loc[both].groupby('borough')['own_matches_crash'].mean().round(3))
print()
print(d.loc[both & ~d['own_matches_crash'], ['borough', 'own_zip', 'crash_zip']].sample(10, random_state=1))

In [ ]:
# Evidence: how many housing rows are follow-up filings?
raw = df_housing_raw.drop_duplicates('job_filing_number').copy()
raw['suffix'] = raw['job_filing_number'].str.extract(r'-([A-Z]+)\d*$')[0]
raw['is_new'] = raw['job_type'].str.contains('New Building', case=False)
print('share of filings that are initial (-I):', f"{(raw['suffix'] == 'I').mean():.1%}")
print('jobs with more than one -I filing:',
      (raw[raw['suffix'] == 'I'].groupby(raw['job_filing_number'].str.split('-').str[0]).size() > 1).sum())
yr = pd.to_datetime(raw['filing_date']).dt.year
print(pd.DataFrame({'all_new_building': raw[raw['is_new']].groupby(yr).size(),
                    'initial_only': raw[raw['is_new'] & (raw['suffix'] == 'I')].groupby(yr).size()}))

In [ ]:
# ===== M1 HOUSING CLEANING (v3: initial applications only, ZIP derived from coordinates) =====
CONTRACT_COLS = ['date', 'year', 'month', 'year_month', 'zip', 'borough', 'lat', 'lon', 'job_type']
MAX_UNITS = 10_000

ZIP_PREFIX_M1 = {'100': 'MANHATTAN', '101': 'MANHATTAN', '102': 'MANHATTAN', '103': 'STATEN ISLAND',
                 '104': 'BRONX', '110': 'QUEENS', '111': 'QUEENS', '112': 'BROOKLYN',
                 '113': 'QUEENS', '114': 'QUEENS', '116': 'QUEENS'}

def zip_matches_borough(z, b):
    """True where the ZIP's first three digits agree with the borough label."""
    z = z.fillna('')
    expected = z.str[:3].map(ZIP_PREFIX_M1)
    nassau_overlap = z.str.startswith('110') & ~z.isin(['11004', '11005'])
    return expected.eq(b) & ~nassau_overlap

steps = []
def log(step, d):
    steps.append((step, len(d)))

h = df_housing_raw.copy()
log('raw download', h)

# 1) Drop the column that is almost half empty and not needed
h = h.drop(columns=['proposed_no_of_stories'])

# 2) Duplicates: exact copies, then repeated filing numbers (keep latest status)
h['current_status_date'] = pd.to_datetime(h['current_status_date'], errors='coerce')
h = h.drop_duplicates()
log('exact duplicate rows removed', h)
h = (h.sort_values(['current_status_date', 'job_filing_number'], na_position='first')
       .drop_duplicates('job_filing_number', keep='last'))
log('repeated job_filing_number removed (latest status kept)', h)

# 2b) Keep initial applications only: later filings on the same job (suffixes P, S, A, Z...) are
#     follow-ups, not new projects, and made counts grow only because amendments pile up.
h['filing_suffix'] = h['job_filing_number'].str.extract(r'-([A-Z]+)\d*$')[0]
h = h[h['filing_suffix'] == 'I'].drop(columns='filing_suffix')
log('follow-up filings removed (kept initial applications only)', h)

# 3) Dates
h = add_time_cols(h, 'filing_date')
h = h.dropna(subset=['date'])
log('rows with unparseable date dropped', h)

# 4) Borough
h['borough'] = clean_borough(h['borough'])
h = h.dropna(subset=['borough'])
log('rows with invalid borough dropped', h)

# 5) Coordinates (needed to derive the ZIP)
h['lat'], h['lon'] = clean_coords(h['latitude'], h['longitude'])
h = h.drop(columns=['latitude', 'longitude'])
h = h.dropna(subset=['lat', 'lon'])
log('rows without valid coordinates dropped', h)

# 6) ZIP: the raw 'zip' column is NOT the property ZIP (only 3.5% match the nearest crash point's ZIP),
#    so keep it for documentation and derive the real ZIP from the coordinates.
h['zip_as_filed'] = clean_zip(h['zip'].astype(str).str.strip())
h['zip'] = np.nan
ref = pd.DataFrame({'zip': clean_zip(df_transport_raw['zip_code'].astype(str)),
                    'lat': pd.to_numeric(df_transport_raw['latitude'], errors='coerce'),
                    'lon': pd.to_numeric(df_transport_raw['longitude'], errors='coerce')})
ref['lat'], ref['lon'] = clean_coords(ref['lat'], ref['lon'])
ref = ref.dropna()
h = fill_zip_nearest(h, ref, max_km=0.5)
print(f"ZIP derived from coordinates: {h['zip'].notna().sum():,} | no crash point within 0.5 km: {h['zip'].isna().sum():,}")
h = h.dropna(subset=['zip'])
log('rows with no crash point within 0.5 km dropped', h)

# 7) Consistency check: derived ZIP must agree with the borough label
bad = ~zip_matches_borough(h['zip'], h['borough'])
print(f"Derived ZIP contradicts borough: {bad.sum():,} rows (dropped)")
h = h[~bad]
log('rows where derived ZIP contradicts borough dropped', h)

# 8) Job type: trim spaces only, keep published labels
h['job_type'] = h['job_type'].astype(str).str.strip()

# 9) Are the missing dwelling units structural? (missing % by job type)
print('\nMissing % by job type:')
print(h.groupby('job_type')[['existing_dwelling_units', 'proposed_dwelling_units']]
        .agg(lambda s: s.isna().mean() * 100).round(1))

# 10) Outliers
for c in ['existing_dwelling_units', 'proposed_dwelling_units']:
    h[c] = pd.to_numeric(h[c], errors='coerce')
    too_big = h[c] > MAX_UNITS
    print(f"{c}: {too_big.sum():,} values above {MAX_UNITS:,} set to missing (largest was {h[c].max():,.0f})")
    h.loc[too_big, c] = np.nan

h['initial_cost'] = pd.to_numeric(h['initial_cost'], errors='coerce')
zero_cost = h['initial_cost'] <= 0
print(f"initial_cost: {zero_cost.sum():,} zero-cost filings set to missing (not reported)")
h.loc[zero_cost, 'initial_cost'] = np.nan
log_cost = np.log10(h['initial_cost'])
lo, hi = iqr_bounds(log_cost.dropna())
h['cost_outlier'] = (log_cost < lo) | (log_cost > hi)
print(f"initial_cost IQR fences (log scale): ${10**lo:,.0f} to ${10**hi:,.0f}; flagged {h['cost_outlier'].sum():,} filings (kept)")

# 11) Final tables
df_housing_full = h.reset_index(drop=True)
df_housing = df_housing_full[CONTRACT_COLS].copy()

funnel = pd.DataFrame(steps, columns=['step', 'rows'])
funnel['removed'] = (-funnel['rows'].diff()).fillna(0).astype(int)
display(funnel)
print(df_housing.shape)
df_housing.head()

### Cleaning decisions (M1)

- **Dates.** `filing_date` was parsed with `pd.to_datetime`; all values parsed. We created `year`, `month` and `year_month` so the table matches the schema contract.
- **Duplicates.** 3 rows were exact copies and were removed. 146 further rows repeated a `job_filing_number`; a filing number identifies one application, so we kept the row with the latest `current_status_date` (the most up-to-date status). Keeping the first row was rejected because it may hold an outdated status.
- **Follow-up filings.** Each job has exactly one initial application (filing number ending in -I; no job has two), and later filings on the same job carry other suffixes (P, S, A, Z and others). Only 51.3% of the distinct filings are initial applications. Counting follow-ups made new-building filings look like they doubled (9,195 in 2022 to 17,566 in 2025) while initial new-building applications were roughly flat (2,483 to 2,386), because follow-ups pile up on earlier projects. We kept initial applications only (357,277 follow-up rows removed), so each row is one application. Limitation: jobs first filed before 2022 are not in the data.
- **Borough.** Names were in title case; we upper-cased them with the shared `clean_borough` so the join key matches M2 and M3. No invalid boroughs were found.
- **Coordinates.** Latitude and longitude were checked with the shared `clean_coords` (NYC bounding box); 1,700 filings without valid coordinates were dropped because the ZIP is derived from them.
- **ZIP (derived from coordinates).** The published `zip` column does not describe the building's location (3.5% match, see the ZIP check). We ignored it (kept as `zip_as_filed`) and assigned each filing the ZIP of the nearest crash record within 0.5 km using the shared `fill_zip_nearest`. 3,702 filings had no crash point within 0.5 km and were dropped. We then checked that the ZIP's first three digits agree with the borough and dropped 725 filings that contradicted it. Limitation: this borrows the crash table's ZIP labels, so a filing near a ZIP boundary can land in the neighbouring ZIP. Staten Island keeps only 79.8% of its initial filings (16,169 of 20,255) against 99.1% to 99.8% elsewhere, so Staten Island results are less reliable.
- **Missing values.** `proposed_no_of_stories` (47.5% missing) was dropped: it is half empty and not needed. Dwelling-unit columns are blank by nature for some job types (New Building has no existing units, Full Demolition has no proposed units, No Work has neither), so we did not impute: a median would invent units that do not exist. They stay missing and are not part of the integration columns.
- **Outliers.** The IQR rule is unsuitable for dwelling units because they are heavily right-skewed, so it would flag genuine large buildings. We used a domain rule instead: more than 10,000 units in one filing is treated as a placeholder or typo and set to missing (15 existing-unit values, largest 111,111,111, and 13 proposed-unit values, largest 294,294). For `initial_cost`, a cost of 0 means not reported and was set to missing (30,996 filings); the remaining costs were checked with IQR on a log10 scale (fences about $154 to $4.9M) and 10,892 filings were flagged in `cost_outlier` but kept, because very expensive projects are real. The histogram also shows a small cluster of filings at about $1, which looks like a placeholder; these fall below the lower fence, are flagged and kept, and do not affect the integration because cost is not used there.
- **Job type.** Labels were kept as published (spaces trimmed), including No Work and Full Demolition, so the integration grouping sees the original text.
- **Result.** 370,870 of the 734,423 downloaded rows (50.5%) were kept, which is 98.4% of the 376,997 initial applications. The table covers 205 ZIPs (Manhattan 68, Queens 62, Brooklyn 39, Bronx 25, Staten Island 11). `df_housing` holds exactly the nine agreed columns; 51,153 of its rows look identical because it has no filing ID, but they are different filings (`df_housing_full` has no repeated filing number) and keeps the extra columns (cost, dwelling units, building type).

In [ ]:
assert list(df_housing.columns) == CONTRACT_COLS, 'columns differ from contract'
assert df_housing[['date', 'zip', 'borough', 'job_type']].isna().sum().sum() == 0, 'nulls in key columns'
assert df_housing['zip'].str.fullmatch(r'\d{5}').all(), 'ZIP is not 5 digits'
assert set(df_housing['borough']) <= set(BOROUGHS), 'unexpected borough'
assert not df_housing_full['job_filing_number'].duplicated().any(), 'duplicate filings remain'
assert df_housing_full['job_filing_number'].str.contains(r'-I\d*$').all(), 'non-initial filings remain'
print('All checks passed')
print('Rows kept:', f"{len(df_housing):,}", 'of', f"{len(df_housing_raw):,}")
print('Distinct ZIPs:', df_housing['zip'].nunique())
print(df_housing.groupby('borough')['zip'].nunique())
print(df_housing['job_type'].value_counts())

base = df_housing_raw.drop_duplicates('job_filing_number')
base = base[base['job_filing_number'].str.extract(r'-([A-Z]+)\d*$')[0] == 'I']
raw_b, kept_b = clean_borough(base['borough']).value_counts(), df_housing['borough'].value_counts()
print(pd.DataFrame({'initial_raw': raw_b, 'kept': kept_b, 'kept_%': (kept_b / raw_b * 100).round(1)}))

fig, ax = plt.subplots(1, 3, figsize=(17, 4))
lc_ = np.log10(df_housing_full['initial_cost'].dropna())
lc_.plot.hist(bins=60, ax=ax[0]); lo_, hi_ = iqr_bounds(lc_)
for v in (lo_, hi_): ax[0].axvline(v, color='red', ls='--')
ax[0].set_title('initial_cost (log10) with IQR fences'); ax[0].set_xlabel('log10 dollars')
funnel[funnel['removed'] > 0].set_index('step')['removed'].plot.barh(ax=ax[1], title='Rows removed per cleaning step (log scale)')
ax[1].set_xscale('log')
df_housing_full['proposed_dwelling_units'].dropna().clip(upper=500).plot.hist(bins=50, ax=ax[2], title='Proposed units (capped at 500 for display)')
plt.tight_layout(); plt.show()

**Validation plots (M1).** Left: the log-scale cost distribution with the IQR fences (red) used to flag cost outliers. Middle: rows removed at each cleaning step (log scale; the follow-up filings step removes by far the most). Right: proposed dwelling units capped at 500, showing the right skew that made the IQR rule unsuitable for them.

In [ ]:
print('--- contract ---')
print(list(df_housing.columns) == CONTRACT_COLS, df_housing.shape)
print(df_housing.dtypes.to_string())
print('\n--- nulls ---'); print(df_housing.isna().sum().to_string())
print('\n--- duplicates ---')
print('exact duplicate rows in df_housing:', df_housing.duplicated().sum())
print('exact duplicate rows in df_housing_full:', df_housing_full.duplicated().sum())
print('repeated job_filing_number:', df_housing_full['job_filing_number'].duplicated().sum())
print('\n--- ranges ---')
print('dates:', df_housing['date'].min(), '->', df_housing['date'].max())
print('years:', sorted(df_housing['year'].unique()), '| months:', sorted(df_housing['month'].unique()))
print('lat:', df_housing['lat'].min(), df_housing['lat'].max(), '| lon:', df_housing['lon'].min(), df_housing['lon'].max())
print('\n--- categories ---')
print(df_housing['borough'].value_counts().to_string())
print(df_housing['job_type'].value_counts().to_string())
print('zip lengths:', df_housing['zip'].astype(str).str.len().value_counts().to_dict(), '| unique zips:', df_housing['zip'].nunique())
print('zip vs borough mismatches:', (~zip_matches_borough(df_housing['zip'], df_housing['borough'])).sum())
print('\n--- numeric (full table) ---')
for c in ['initial_cost', 'proposed_dwelling_units']:
    if c in df_housing_full:
        print(c, df_housing_full[c].describe().round(0).to_dict())
print('\n--- funnel ---'); print(funnel.to_string())

## 4. TRANSPORT: EDA and pre-integration cleaning (**M2**)

In [ ]:
overview(df_transport_raw, 'Transport raw')
df_transport_raw.describe(include='all').T.head(30)

In [ ]:
# 4a. EDA step 1: structure, duplicates, dates, borough, ZIP, coordinates
t = df_transport_raw
print('Duplicate collision_id:', t['collision_id'].duplicated().sum())
print('Fully duplicated rows :', t.duplicated().sum())
print()

dates = pd.to_datetime(t['crash_date'], errors='coerce')
print('Unparseable dates:', dates.isna().sum())
print('Date range       :', dates.min(), '->', dates.max())
print()

print(t['borough'].value_counts(dropna=False))
print()

zips = clean_zip(t['zip_code'])
print(f"ZIP missing in raw                 : {t['zip_code'].isna().mean():.1%}")
print(f"ZIP missing/invalid after clean_zip: {zips.isna().mean():.1%}")
print()

lat = pd.to_numeric(t['latitude'], errors='coerce')
lon = pd.to_numeric(t['longitude'], errors='coerce')
clat, clon = clean_coords(lat, lon)
print(f"Coordinates missing        : {lat.isna().mean():.1%}")
print(f"Coordinates exactly 0,0    : {((lat == 0) & (lon == 0)).mean():.2%}")
print(f"Invalid after clean_coords : {clat.isna().mean():.1%}")
print()

# no ZIP but valid coordinates = recoverable with fill_zip_nearest; no ZIP and no coordinates = lost
print(f"No ZIP but valid coordinates (recoverable): {(zips.isna() & clat.notna()).sum():,}")
print(f"No ZIP and no coordinates (lost)          : {(zips.isna() & clat.isna()).sum():,}")

df_transport = df_transport_raw.copy()

## Structure (M2)
The raw table has 448,091 crashes. `collision_id` is unique, no rows are duplicated, all dates parse, and the date range ends on 3 Nov 2026. 28.3% of crashes have no valid ZIP, 101,291 (22.6%) have no borough, and 6.9% have unusable coordinates (5.5% missing, 1.41% at 0,0). Of the crashes without a ZIP, 105,267 have valid coordinates and can be recovered; 21,465 have neither.

In [ ]:
# 4b. EDA step 2: count columns, consistency, and does missing ZIP matter?
count_cols = ['number_of_persons_injured', 'number_of_persons_killed',
              'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
              'number_of_cyclist_injured', 'number_of_cyclist_killed',
              'number_of_motorist_injured', 'number_of_motorist_killed']
nums = df_transport_raw[count_cols]

print(nums.describe(percentiles=[.5, .9, .99, .999]).round(2).T)
print('\nNegative values:', (nums < 0).sum().sum())

# consistency: person totals vs the sum of the three groups
inj_parts  = nums[['number_of_pedestrians_injured', 'number_of_cyclist_injured', 'number_of_motorist_injured']].sum(axis=1)
kill_parts = nums[['number_of_pedestrians_killed', 'number_of_cyclist_killed', 'number_of_motorist_killed']].sum(axis=1)
print(f"Injured total < sum of groups: {(nums['number_of_persons_injured'] < inj_parts).sum():,}")
print(f"Killed total < sum of groups : {(nums['number_of_persons_killed'] < kill_parts).sum():,}")

# are crashes with a missing ZIP different from the rest?
zip_missing = clean_zip(df_transport_raw['zip_code']).isna().rename('zip_missing')
sev = pd.DataFrame({'injured_per_crash': nums['number_of_persons_injured'],
                    'killed_per_crash': nums['number_of_persons_killed'],
                    'share_with_injury': nums['number_of_persons_injured'] > 0}).groupby(zip_missing).mean().round(3)
print('\nSeverity, ZIP present (False) vs missing (True):')
print(sev)

# last months in the data (checks the partial last month)
print('\nCrashes in the last 5 months:')
print(pd.to_datetime(df_transport_raw['crash_date']).dt.to_period('M').value_counts().sort_index().tail(5))

## Injury and fatality counts.
All eight columns are complete integers with no negatives, and no person total is smaller than the sum of its pedestrian, cyclist and motorist counts. They are very right-skewed (median 0, maximum 40 injured in one crash).
## Missing ZIPs are not random.
Crashes without a ZIP are more severe: 0.651 injuries per crash against 0.528, and 44.5% with an injury against 41.0%. Dropping them would make the data look safer than it is, so we recover the ZIP from coordinates first.
## Date coverage.
November 2026 has only 594 crashes (Jul to Oct 2026 have 6,879 to 7,704) because the data stops on 3 Nov, so it is not a complete month.

In [ ]:
# 4c. EDA step 3: plots
hour = pd.to_datetime(df_transport_raw['crash_time'], format='%H:%M', errors='coerce').dt.hour
fig, ax = plt.subplots(3, 2, figsize=(15, 14))

df_transport_raw['borough'].fillna('MISSING').astype(str).str.upper().value_counts().plot.bar(ax=ax[0, 0], title='Crashes by borough', rot=30)

hour.value_counts().sort_index().plot.bar(ax=ax[0, 1], title='Crashes by hour of day')

dates.dt.day_name().value_counts().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
).plot.bar(ax=ax[1, 0], title='Crashes by weekday', rot=30)

monthly = dates.dt.to_period('M').value_counts().sort_index()
monthly.index = monthly.index.to_timestamp()
monthly.plot(ax=ax[1, 1], title='Crashes per month')

nums['number_of_persons_injured'].clip(upper=6).value_counts().sort_index().plot.bar(
    ax=ax[2, 0], title='Persons injured per crash (6 = 6 or more)', logy=True)

miss = df_transport_raw.isna().mean().mul(100).sort_values(ascending=False)
miss[miss > 0].plot.barh(ax=ax[2, 1], title='Missing values (%)')
ax[2, 1].invert_yaxis()

plt.tight_layout()
plt.show()

## Plots (M2).
Brooklyn has the most crashes (about 118K) and Staten Island the fewest (about 14K); the 101,291 crashes with no borough would rank second. Crashes peak between 2 and 5 p.m., with a smaller spike at 8 a.m., and are fairly even across weekdays. Monthly crashes drift down from about 8K to 9K in 2022 to about 7K in 2025 to 2026, and about 250K of the 448K crashes injure nobody.

In [ ]:
# 4d. CLEANING step 1: working copy, row log, duplicates guard, dates
CONTRACT_COLS_T = ['date', 'year', 'month', 'year_month', 'zip', 'borough', 'lat', 'lon',
                   'number_of_persons_injured', 'number_of_persons_killed',
                   'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
                   'number_of_cyclist_injured', 'number_of_cyclist_killed',
                   'number_of_motorist_injured', 'number_of_motorist_killed']
steps_t = []
def log_t(step, d):
    steps_t.append((step, len(d)))

tr = df_transport_raw.copy()
log_t('raw download', tr)

tr = tr.drop_duplicates('collision_id')
log_t('repeated collision_id removed', tr)

tr = add_time_cols(tr, 'crash_date')
tr = tr.dropna(subset=['date'])
log_t('rows with unparseable date dropped', tr)

print(steps_t)
print(tr[['date', 'year', 'month', 'year_month']].head())
print(tr['year'].value_counts().sort_index())

## Dates and duplicates (M2)
`crash_date` was parsed with the shared `add_time_cols`, which created `year`, `month` and `year_month`. No dates failed and no `collision_id` was repeated, so no rows were removed; the check is kept so the cleaning still works if the data is downloaded again.

In [ ]:
# 4e. CLEANING step 2: borough
before = tr['borough'].isna().sum()
tr['borough'] = clean_borough(tr['borough'])
print('Borough values:', tr['borough'].value_counts(dropna=False).to_dict())
print(f"Missing before: {before:,} | after clean_borough: {tr['borough'].isna().sum():,}")

## Borough (M2)
Borough labels were standardised with the shared `clean_borough` (upper case, the five valid names only); no label was changed, and the 101,291 crashes without a borough stay missing. They are filled from the ZIP prefix in a later step, the same rule used in section 6.

In [ ]:
# 4f. CLEANING step 3: coordinates
lat0 = pd.to_numeric(tr['latitude'], errors='coerce')
lon0 = pd.to_numeric(tr['longitude'], errors='coerce')
tr['lat'], tr['lon'] = clean_coords(lat0, lon0)
tr = tr.drop(columns=['latitude', 'longitude'])

n_missing = lat0.isna().sum()
n_zero = ((lat0 == 0) & (lon0 == 0)).sum()
n_invalid = tr['lat'].isna().sum()
print(f"Missing: {n_missing:,} | exactly 0,0: {n_zero:,} | other outside NYC box: {n_invalid - n_missing - n_zero:,}")
print(f"Coordinates now missing: {n_invalid:,} ({n_invalid / len(tr):.1%})")

## Coordinates (M2).
Latitude and longitude were checked with the shared `clean_coords` (NYC bounding box). 24,532 were missing and 6,303 sat at the 0,0 placeholder; none fell outside the box otherwise, so 30,835 crashes (6.9%) have no usable coordinates. We kept these rows rather than dropping them: 21,465 of them also have no ZIP (from the EDA), but the other 9,370 still have one. They only cannot be used to recover a missing ZIP.

In [ ]:
# 4g. CLEANING step 4a: how accurate is "nearest crash within 0.5 km" at recovering a ZIP?
tr['zip'] = clean_zip(tr['zip_code'])

known = tr.dropna(subset=['zip', 'lat', 'lon'])
test = known.sample(20_000, random_state=1)
ref_t = known.drop(test.index)

tree = BallTree(np.radians(ref_t[['lat', 'lon']].to_numpy()), metric='haversine')
dist, idx = tree.query(np.radians(test[['lat', 'lon']].to_numpy()), k=1)
within = dist[:, 0] * 6371.0088 <= 0.5
pred = ref_t['zip'].to_numpy()[idx[:, 0]]

print(f"Test sample: {len(test):,} crashes with a known ZIP")
print(f"Nearest other crash within 0.5 km: {within.mean():.1%}")
print(f"Recovered ZIP equals the true ZIP (where within 0.5 km): {(pred[within] == test['zip'].to_numpy()[within]).mean():.1%}")

## Testing the ZIP recovery (M2)
To check the method, we hid the ZIP of 20,000 crashes that have one and recovered it from the nearest other crash within 0.5 km. The recovered ZIP matched the true one in 99.1% of cases. This test used crashes that already had a ZIP, so it may overstate the accuracy for crashes that really lack one.

In [ ]:
# 4h. CLEANING step 4b: recover missing ZIPs from coordinates, drop the rest
tr = tr.drop(columns=['zip_code'])
missing_before = tr['zip'].isna().sum()

tr = fill_zip_nearest(tr, tr, max_km=0.5)
missing_after = tr['zip'].isna().sum()

print(f"ZIP missing before: {missing_before:,}")
print(f"Recovered from coordinates: {missing_before - missing_after:,}")
print(f"Still missing: {missing_after:,}")
print(f"  no valid coordinates : {(tr['zip'].isna() & tr['lat'].isna()).sum():,}")
print(f"  coordinates but no crash with a ZIP within 0.5 km: {(tr['zip'].isna() & tr['lat'].notna()).sum():,}")

tr = tr.dropna(subset=['zip'])
log_t('rows with no ZIP after recovery dropped', tr)
print(f"Kept {len(tr):,} of {len(df_transport_raw):,} ({len(tr) / len(df_transport_raw):.1%})")

## ZIP (M2).
The shared `clean_zip` kept 5-digit NYC ZIPs only. Of 126,732 crashes without a valid ZIP, 99,144 were recovered with `fill_zip_nearest` (nearest crash with a ZIP within 0.5 km). We recover instead of dropping because crashes without a ZIP are more severe (see the EDA). The remaining 27,588 were dropped because ZIP is the join key: 21,465 have no coordinates and 6,123 have no crash with a ZIP within 0.5 km. This keeps 420,503 of 448,091 crashes (93.8%). Limitation: recovered ZIPs are borrowed from other crashes, so a crash near a ZIP boundary can land in the neighbouring ZIP.

In [ ]:
# 4i. CLEANING step 5a: do ZIPs and borough labels agree?
ZIP_PREFIX_T = {'100': 'MANHATTAN', '101': 'MANHATTAN', '102': 'MANHATTAN', '103': 'STATEN ISLAND',
                '104': 'BRONX', '110': 'QUEENS', '111': 'QUEENS', '112': 'BROOKLYN',
                '113': 'QUEENS', '114': 'QUEENS', '116': 'QUEENS'}
QUEENS_110 = ['11004', '11005']    # the only 110xx ZIPs inside NYC; the rest of 110xx is Nassau County

def zip_to_borough(z):
    b = z.str[:3].map(ZIP_PREFIX_T)
    return b.where(~(z.str.startswith('110') & ~z.isin(QUEENS_110)))

tr['borough_from_zip'] = zip_to_borough(tr['zip'])
has_label = tr['borough'].notna()

print(f"ZIPs whose prefix is not a NYC borough prefix: {tr['borough_from_zip'].isna().sum():,}")
print(tr.loc[tr['borough_from_zip'].isna(), 'zip'].value_counts().head(10))
print()
print(f"Crashes with a borough label: {has_label.sum():,}")
agree = tr.loc[has_label, 'borough'] == tr.loc[has_label, 'borough_from_zip']
print(f"  label agrees with the ZIP-prefix borough: {agree.sum():,} ({agree.mean():.2%})")
print(tr.loc[has_label].loc[~agree].groupby(['borough', 'borough_from_zip'], dropna=False).size())
print()
print(f"Crashes without a label that the ZIP can fill: {(~has_label & tr['borough_from_zip'].notna()).sum():,}")

## Borough versus ZIP (M2).
Of the 345,386 crashes that have both a borough label and a ZIP, 99.88% (344,969) agree with the borough implied by the ZIP prefix, so the prefix is reliable for filling the missing boroughs. 417 crashes disagree: 196 point to a different NYC borough, and 221 have a Queens label but a ZIP outside the NYC prefix list (11001 or 11040; 240 crashes in total have one of these two ZIPs).

In [ ]:
# 4j. CLEANING step 5b: borough from ZIP prefix; drop non-NYC ZIPs and contradictions
non_nyc = tr['borough_from_zip'].isna()
print(f"ZIP outside the NYC prefixes (11001, 11040): {non_nyc.sum():,} crashes dropped")
tr = tr[~non_nyc]
log_t('rows with a ZIP outside the NYC prefixes dropped', tr)

clash = tr['borough'].notna() & (tr['borough'] != tr['borough_from_zip'])
print(f"Borough label contradicts the ZIP: {clash.sum():,} crashes dropped")
tr = tr[~clash]
log_t('rows where the borough label contradicts the ZIP dropped', tr)

filled = tr['borough'].isna().sum()
tr['borough'] = tr['borough'].fillna(tr['borough_from_zip'])
tr = tr.drop(columns=['borough_from_zip'])
print(f"Borough filled from the ZIP prefix: {filled:,} | borough missing now: {tr['borough'].isna().sum():,}")
print(tr['borough'].value_counts())
print(f"Kept {len(tr):,} of {len(df_transport_raw):,} ({len(tr) / len(df_transport_raw):.1%})")

## Borough from ZIP (M2).
240 crashes with ZIP 11001 or 11040 (outside the NYC prefixes) and 196 whose borough label contradicts their ZIP were dropped, about 0.1% of the table; the ZIP is the join key, so a crash we cannot place reliably is not worth keeping. The 75,098 crashes without a borough were then filled from the ZIP prefix, which agreed with the label in 99.88% of labelled crashes, so no borough is missing. After the ZIP and borough steps 420,067 of 448,091 crashes remain (93.7%).

In [ ]:
# 4k. CLEANING step 6: outliers in the injury and fatality counts
for col in ['number_of_persons_injured', 'number_of_persons_killed']:
    lo, hi = iqr_bounds(tr[col])
    n_out = (tr[col] > hi).sum()
    print(f"{col}: IQR fences {lo:.1f} to {hi:.1f} -> {n_out:,} crashes above ({n_out / len(tr):.1%})")

inj_parts = tr[['number_of_pedestrians_injured', 'number_of_cyclist_injured', 'number_of_motorist_injured']].sum(axis=1)
print(f"\nInjured total > sum of the three groups: {(tr['number_of_persons_injured'] > inj_parts).sum():,}")
print(f"Crashes with 10 or more injured: {(tr['number_of_persons_injured'] >= 10).sum():,}")
print('\nLargest crashes by persons injured:')
print(tr.nlargest(5, 'number_of_persons_injured')[['date', 'borough', 'zip', 'number_of_persons_injured',
                                                    'number_of_pedestrians_injured', 'number_of_cyclist_injured',
                                                    'number_of_motorist_injured']].to_string(index=False))

## Outliers (M2).
No injury or fatality counts were removed. The IQR rule does not fit them: with a median of 0, its upper fence is 2.5 injured, which flags 12,797 crashes (3.0%), and for deaths the fence is 0, which flags all 1,077 fatal crashes (0.3%). The largest crash has 34 injured, all counted as motorists, and 90 crashes have 10 or more; for the five largest, the group counts add up to the total, so nothing marks them as errors. In 11,942 crashes the injured total is higher than the pedestrian, cyclist and motorist counts combined, so those three groups do not cover every injured person; we keep the published total.

In [ ]:
# 4l. CLEANING step 7: final tables, checks, row funnel
df_transport_full = tr.reset_index(drop=True)
df_transport = df_transport_full[CONTRACT_COLS_T].copy()

KEY_T = ['date', 'year', 'month', 'year_month', 'zip', 'borough']
COUNTS_T = CONTRACT_COLS_T[8:]
assert list(df_transport.columns) == CONTRACT_COLS_T, 'columns differ from contract'
assert df_transport[KEY_T + COUNTS_T].isna().sum().sum() == 0, 'nulls in key or count columns'
assert df_transport['zip'].str.fullmatch(r'\d{5}').all(), 'ZIP is not 5 digits'
assert set(df_transport['borough']) <= set(BOROUGHS), 'unexpected borough'
assert (df_transport[COUNTS_T] >= 0).all().all(), 'negative counts'
assert not df_transport_full['collision_id'].duplicated().any(), 'duplicate collisions remain'

print('All checks passed')
print('Rows kept:', f"{len(df_transport):,}", 'of', f"{len(df_transport_raw):,}")
print('Distinct ZIPs:', df_transport['zip'].nunique())
print(df_transport.groupby('borough')['zip'].nunique())
print('Rows with no coordinates (lat/lon missing):', f"{df_transport['lat'].isna().sum():,}")

funnel_t = pd.DataFrame(steps_t, columns=['step', 'rows'])
funnel_t['removed'] = (-funnel_t['rows'].diff()).fillna(0).astype(int)
display(funnel_t)

## Result (M2).
420,067 of 448,091 crashes (93.7%) were kept. The 28,024 removed are 27,588 with no ZIP after recovery, 240 with a ZIP outside NYC and 196 whose borough label contradicted the ZIP. The table covers 226 ZIPs (Manhattan 87, Queens 64, Brooklyn 39, Bronx 25, Staten Island 11) and has no missing values in the date, ZIP, borough or count columns; 9,355 rows have a ZIP but no coordinates. `df_transport` holds exactly the agreed columns, and `df_transport_full` keeps the extras (`collision_id`, `crash_time`, `on_street_name`) for our own analysis.

## 5. INTEGRATION (**M3**)
### 5a. Quick-clean stand-ins
These give a first working `merged_data.csv` today, so the website can be built on real data while M1 and M2 do their full cleaning.

In [ ]:
def quick_clean_transport(raw):
    """Minimal standardisation so integration can run NOW. M2 replaces this with the full cleaning
    (EDA, outliers, justified decisions) as long as the OUTPUT COLUMNS stay the same."""
    d = raw.copy()
    n0 = len(d)
    if 'collision_id' in d:
        d = d.drop_duplicates('collision_id')
    d = add_time_cols(d, 'crash_date')
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['zip'] = clean_zip(d['zip_code'])
    out['borough'] = clean_borough(d['borough'])
    out['lat'], out['lon'] = clean_coords(d['latitude'], d['longitude'])
    for c in ['number_of_persons_injured', 'number_of_persons_killed',
              'number_of_pedestrians_injured', 'number_of_pedestrians_killed',
              'number_of_cyclist_injured', 'number_of_cyclist_killed',
              'number_of_motorist_injured', 'number_of_motorist_killed']:
        out[c] = pd.to_numeric(d[c], errors='coerce').fillna(0) if c in d else 0
    missing_before = out['zip'].isna().sum()
    out = fill_zip_nearest(out, out)                       # recover ZIP from the nearest crash that has one
    print(f'[transport] rows {n0:,} -> {len(d):,} after dedupe | ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after coordinate recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[transport] kept {len(out):,} rows with valid date + ZIP')
    return out

def quick_clean_housing(raw, zip_reference=None):
    """Minimal standardisation. M1 replaces it with the full cleaning (same output columns).
    zip_reference: cleaned transport table, used to recover ZIPs if the housing table lacks them."""
    cols = raw.columns
    date_col = find_col(cols, ['filing_date', 'filed_date'], regex=r'filing.*date|date.*filing')
    type_col = find_col(cols, ['job_type'], regex=r'job.*type')
    zip_col  = find_col(cols, ['zip_code', 'zip', 'zipcode'], regex=r'^zip', required=False)
    lat_col  = find_col(cols, ['latitude', 'gis_latitude'], regex=r'^(gis_)?lat', required=False)
    lon_col  = find_col(cols, ['longitude', 'gis_longitude'], regex=r'^(gis_)?lon', required=False)
    id_col   = find_col(cols, ['job_filing_number'], required=False)
    d = raw.copy()
    n0 = len(d)
    if id_col:
        d = d.drop_duplicates(id_col)
    d = add_time_cols(d, date_col)
    out = pd.DataFrame(index=d.index)
    out['date'], out['year'], out['month'], out['year_month'] = d['date'], d['year'], d['month'], d['year_month']
    out['job_type'] = d[type_col]
    out['zip'] = clean_zip(d[zip_col]) if zip_col else np.nan
    out['borough'] = clean_borough(d['borough']) if 'borough' in d else np.nan
    if lat_col and lon_col:
        out['lat'], out['lon'] = clean_coords(d[lat_col], d[lon_col])
    else:
        out['lat'], out['lon'] = np.nan, np.nan
    missing_before = out['zip'].isna().sum()
    if zip_reference is not None:
        out = fill_zip_nearest(out, zip_reference)
    print(f'[housing] rows {n0:,} -> {len(d):,} after dedupe | ZIP column found: {bool(zip_col)} | '
          f'ZIP missing {missing_before:,} -> {out["zip"].isna().sum():,} after recovery')
    out = out.dropna(subset=['date', 'zip'])
    print(f'[housing] kept {len(out):,} rows with valid date + ZIP')
    return out

In [ ]:
df_transport_q = quick_clean_transport(df_transport_raw)
df_housing_q   = quick_clean_housing(df_housing_raw, zip_reference=df_transport_q)

# Flip to True once M1 and M2 have delivered their full cleaning (same columns, see the contract above)
USE_FULL_CLEANING = True
housing_for_join   = df_housing   if USE_FULL_CLEANING else df_housing_q
transport_for_join = df_transport if USE_FULL_CLEANING else df_transport_q

### 5b. Aggregate and join at ZIP x month

In [ ]:
SCHEMA_COLS = ['zip', 'borough', 'year', 'month', 'year_month', 'lat', 'lon',
               'filings_new_building', 'filings_alteration', 'filings_other',
               'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
               'injuries', 'fatalities']

def housing_group(job_type):
    t = job_type.astype(str).str.strip().str.lower()
    is_nb  = t.str.contains('new building') | t.eq('nb')
    is_alt = t.str.contains('alteration') | t.str.fullmatch(r'a[123]')
    return np.select([is_nb, is_alt], ['new_building', 'alteration'], default='other')

def aggregate_housing(h):
    """One row per ZIP x month: number of job filings by group."""
    h = h.assign(job_group=housing_group(h['job_type']))
    g = h.groupby(['zip', 'year_month', 'job_group']).size().unstack('job_group', fill_value=0)
    for c in ['new_building', 'alteration', 'other']:
        if c not in g:
            g[c] = 0
    return g[['new_building', 'alteration', 'other']].add_prefix('filings_').reset_index()

def aggregate_transport(t):
    """One row per ZIP x month: crashes (all / with pedestrian / cyclist / motorist casualties), injuries, deaths."""
    t = t.assign(
        ped=(t['number_of_pedestrians_injured'] + t['number_of_pedestrians_killed']) > 0,
        cyc=(t['number_of_cyclist_injured'] + t['number_of_cyclist_killed']) > 0,
        mot=(t['number_of_motorist_injured'] + t['number_of_motorist_killed']) > 0)
    return (t.groupby(['zip', 'year_month'])
              .agg(crashes_total=('ped', 'size'), crashes_pedestrian=('ped', 'sum'),
                   crashes_cyclist=('cyc', 'sum'), crashes_motorist=('mot', 'sum'),
                   injuries=('number_of_persons_injured', 'sum'),
                   fatalities=('number_of_persons_killed', 'sum')).reset_index())

def build_zip_month_table(housing, transport):
    """Integrate at ZIP x month. Decisions (justify in the markdown cell below):
       1. keep only ZIPs present in BOTH datasets (a ZIP seen on one side only gives no comparison)
       2. use the common date window and drop the last month if it is incomplete
       3. build a COMPLETE ZIP x month grid so 'no filings / no crashes' is an explicit 0, not a missing row"""
    zips = sorted(set(housing['zip']) & set(transport['zip']))
    h, t = housing[housing['zip'].isin(zips)], transport[transport['zip'].isin(zips)]
    start = max(h['date'].min(), t['date'].min()).to_period('M')
    end_date = min(h['date'].max(), t['date'].max())
    end = end_date.to_period('M')
    if end_date.normalize() < end.end_time.normalize():     # last month not finished -> drop it
        end = end - 1
    months = [str(p) for p in pd.period_range(start, end, freq='M')]
    h, t = h[h['year_month'].isin(months)], t[t['year_month'].isin(months)]

    grid = pd.MultiIndex.from_product([zips, months], names=['zip', 'year_month']).to_frame(index=False)
    merged = (grid.merge(aggregate_housing(h), on=['zip', 'year_month'], how='left')
                  .merge(aggregate_transport(t), on=['zip', 'year_month'], how='left'))

    # ZIP-level attributes: borough = most common borough label, lat/lon = centroid of all point records
    pts = pd.concat([h[['zip', 'borough', 'lat', 'lon']], t[['zip', 'borough', 'lat', 'lon']]])
    zip_info = pts.groupby('zip').agg(
        borough=('borough', lambda s: s.mode().iloc[0] if s.notna().any() else np.nan),
        lat=('lat', 'mean'), lon=('lon', 'mean')).reset_index()
    merged = merged.merge(zip_info, on='zip', how='left')
    merged['year'] = merged['year_month'].str[:4].astype(int)
    merged['month'] = merged['year_month'].str[5:7].astype(int)

    info = dict(zips_housing=housing['zip'].nunique(), zips_transport=transport['zip'].nunique(),
                zips_kept=len(zips), window=f'{months[0]} to {months[-1]}', months=len(months),
                grid_rows=len(grid), housing_rows_used=len(h), transport_rows_used=len(t))
    return merged, info

In [ ]:
merged, info = build_zip_month_table(housing_for_join, transport_for_join)
print(info)
merged.head()

**Justification (M3: replace the numbers with `info` from your run)**
- **Join key `zip` + `year_month`.** Individual filings and crashes do not correspond to each other, so we compare *activity levels* per neighbourhood-month. ZIP is available on both sides (or recoverable from coordinates) and gives 205 ZIP codes, far more than the 5 boroughs; month keeps seasonality and allows lag analysis.
- **Alternatives considered:** borough x month (only 5 units, hides local patterns); row-level spatial join (a filing has no matching crash row, so it would be meaningless); census tract (too sparse per month).
- **Only ZIPs present in both datasets are kept** (a ZIP seen on one side gives no comparison).
- **Common window, partial last month dropped** (the crash feed ends earlier than the housing feed).
- **Complete ZIP x month grid:** months with no filings or no crashes are real zeros, so they are kept as explicit 0 rows instead of missing.
- **ZIP recovery:** missing crash ZIPs are filled from the nearest crash that has one (<= 0.5 km). If the housing table has no ZIP, it is filled from the nearest crash point: this borrows the crash table's ZIP boundaries, which we list as a limitation.

In [ ]:
from IPython.display import Markdown, display
display(Markdown(f"""**Result of the integration:** {info['zips_kept']} ZIP codes appear in both datasets (housing had {info['zips_housing']} ZIPs, crashes had {info['zips_transport']}), \
over {info['months']} months ({info['window']}), giving a complete grid of {info['grid_rows']:,} ZIP-month rows. \
{info['housing_rows_used']:,} housing filings and {info['transport_rows_used']:,} crashes fall inside that grid."""))

## 6. POST-INTEGRATION CLEANING (**M3**) (required)

In [ ]:
COUNT_COLS = ['filings_new_building', 'filings_alteration', 'filings_other',
              'crashes_total', 'crashes_pedestrian', 'crashes_cyclist', 'crashes_motorist',
              'injuries', 'fatalities']

# The first 3 digits of a NYC ZIP identify the borough reliably, so we use them to catch wrong borough labels
ZIP_PREFIX_BOROUGH = {'100': 'MANHATTAN', '101': 'MANHATTAN', '102': 'MANHATTAN', '103': 'STATEN ISLAND',
                      '104': 'BRONX', '110': 'QUEENS', '111': 'QUEENS', '112': 'BROOKLYN',
                      '113': 'QUEENS', '114': 'QUEENS', '116': 'QUEENS'}

def fix_borough_by_zip(m):
    """Inconsistency check: where the borough label disagrees with the ZIP prefix, trust the ZIP."""
    m = m.copy()
    expected = m['zip'].str[:3].map(ZIP_PREFIX_BOROUGH)
    wrong = expected.notna() & (expected != m['borough'])
    if wrong.any():
        fixes = (m.loc[wrong, ['zip', 'borough']].drop_duplicates('zip').assign(corrected_to=expected[wrong]))
        print(f'Borough corrected from ZIP prefix for {fixes.shape[0]} ZIPs:'); print(fixes.to_string(index=False))
    m.loc[wrong, 'borough'] = expected[wrong]
    return m

def post_integration_clean(merged):
    """1) NaN from the left joins = no activity -> 0   2) fix dtypes   3) drop rows with no borough/coords
       4) remove duplicates   5) fix borough labels that contradict the ZIP   6) enforce the agreed schema/column order"""
    m = merged.copy()
    n0 = len(m)
    print('NaNs per column BEFORE:\n', m.isna().sum()[lambda s: s > 0].to_string())
    m[COUNT_COLS] = m[COUNT_COLS].fillna(0).astype(int)
    m['zip'] = m['zip'].astype(str).str.zfill(5)
    m = m.dropna(subset=['borough', 'lat', 'lon']).drop_duplicates(['zip', 'year_month'])
    m = fix_borough_by_zip(m)
    print(f'rows {n0:,} -> {len(m):,} (dropped rows without borough/coordinates, and duplicates)')
    return m[SCHEMA_COLS].sort_values(['zip', 'year_month']).reset_index(drop=True)

def validate(m):
    """Fail loudly if the table breaks the contract that the website and teammates rely on."""
    assert list(m.columns) == SCHEMA_COLS, 'columns differ from the schema contract'
    assert m.isna().sum().sum() == 0, 'nulls remain'
    assert not m.duplicated(['zip', 'year_month']).any(), 'duplicate ZIP-months'
    assert (m[COUNT_COLS] >= 0).all().all(), 'negative counts'
    assert (m['crashes_pedestrian'] <= m['crashes_total']).all(), 'pedestrian crashes exceed total'
    assert set(m['borough']) <= set(BOROUGHS), 'unexpected borough labels'
    print('VALID:', m.shape, '|', m['zip'].nunique(), 'ZIPs x', m['year_month'].nunique(), 'months')

def outlier_report(m, cols=('filings_new_building', 'crashes_total')):
    """IQR rule per ZIP total. Outliers are REPORTED, not removed: they are real high-activity ZIPs."""
    tot = m.groupby('zip')[list(cols)].sum()
    for c in cols:
        q1, q3 = tot[c].quantile([.25, .75]); hi = q3 + 1.5 * (q3 - q1)
        print(f'{c}: IQR upper bound {hi:,.0f} -> {(tot[c] > hi).sum()} ZIPs above it')

In [ ]:
final = post_integration_clean(merged)     # new NaNs from the joins -> 0, dtypes, redundant rows, schema order
validate(final)                            # fails loudly if the contract is broken
outlier_report(final)
final.describe().T

In [ ]:
# Sanity plots: do the integrated numbers look believable?
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
final.groupby('year_month')[['filings_new_building', 'crashes_total']].sum().plot(ax=ax[0], secondary_y='crashes_total', title='Monthly totals')
z = final.groupby('zip')[['filings_new_building', 'crashes_total']].sum()
sns.scatterplot(data=z, x='filings_new_building', y='crashes_total', ax=ax[1]); ax[1].set_title('ZIP totals')
sns.heatmap(final.pivot_table(index='borough', columns='month', values='crashes_total', aggfunc='sum'), ax=ax[2], cmap='YlOrRd')
ax[2].set_title('Crashes: borough x month')
plt.tight_layout(); plt.show()

In [ ]:
# Export: THIS file is what the website reads (put it in the repo at data/merged_data.csv)
final.to_csv('merged_data.csv', index=False)
try:
    from google.colab import files
    files.download('merged_data.csv')
except ImportError:
    pass
print(final.shape)

### 6b. Data health check (M3): numbers to quote in section 1 and the justification

In [ ]:
def health_report(final, housing_raw, transport_raw, housing_used, transport_used, info):
    """Numbers to quote in section 1 and the justification: row funnel, date windows, zeros, ZIP coverage."""
    print('--- ROW FUNNEL (how many records survived each step) ---')
    print(f"housing  : raw {len(housing_raw):,} -> cleaned {len(housing_used):,} -> inside window {info['housing_rows_used']:,}")
    print(f"transport: raw {len(transport_raw):,} -> cleaned {len(transport_used):,} -> inside window {info['transport_rows_used']:,}")
    print('\n--- DATE WINDOWS ---')
    print('housing  latest date :', housing_used['date'].max().date())
    print('crashes  latest date :', transport_used['date'].max().date(), '<- the crash feed may stop earlier than housing')
    print('integrated window    :', info['window'], f"({info['months']} months)")
    print('\n--- ZIP COVERAGE ---')
    print(f"ZIPs in housing {info['zips_housing']}, in crashes {info['zips_transport']}, kept (in both) {info['zips_kept']}")
    print(final.groupby('borough')['zip'].nunique().rename('ZIPs per borough').to_string())
    print('\n--- SHARE OF ZIP-MONTHS WITH ZERO ACTIVITY (quote this in your limitations) ---')
    for c in ['filings_new_building', 'filings_alteration', 'crashes_total', 'crashes_pedestrian', 'crashes_cyclist']:
        print(f'{c:24s} {(final[c] == 0).mean():6.1%}')
    print('\n--- TOTALS ---')
    print(final[['filings_new_building', 'filings_alteration', 'filings_other', 'crashes_total', 'injuries', 'fatalities']].sum().to_string())

In [ ]:
health_report(final, df_housing_raw, df_transport_raw, housing_for_join, transport_for_join, info)

In [ ]:
from IPython.display import Markdown, display
# Every number below is computed from the data, so it stays correct if M1/M2 later change the cleaning.
n_t = df_transport_raw.drop_duplicates('collision_id').shape[0] if 'collision_id' in df_transport_raw else len(df_transport_raw)
n_h = df_housing_raw.drop_duplicates('job_filing_number').shape[0] if 'job_filing_number' in df_housing_raw else len(df_housing_raw)
unplaced_t = 1 - len(transport_for_join) / n_t
unplaced_h = 1 - len(housing_for_join) / n_h
yr = final.groupby('year')[['filings_new_building', 'filings_alteration']].sum()
full = yr.loc[2022:2025]
display(Markdown(f"""**Limitations of the integrated data (M3)**
- The latest crash record in our download is from {transport_for_join['date'].max():%B %Y}, and the integrated window is {info['window']}.
- Filings are applications, not buildings finished or work started; timing can differ by months.
- {unplaced_t:.0%} of crash records and {unplaced_h:.0%} of housing filings could not be given a valid ZIP and date, and were dropped. Missing ZIPs were first recovered from the nearest record within 0.5 km.
- {(final['filings_new_building'] == 0).mean():.0%} of ZIP-months have zero new-building filings, so new-building results rest on few events in many ZIPs.
- Only ZIPs present in both datasets are kept ({info['zips_kept']} of {info['zips_housing']} housing ZIPs and {info['zips_transport']} crash ZIPs).
- New-building filings rose from {full.loc[2022, 'filings_new_building']:,} (2022) to {full.loc[2025, 'filings_new_building']:,} (2025) while alteration filings stayed between {full['filings_alteration'].min():,} and {full['filings_alteration'].max():,} a year. We do not read this as a construction boom: it may partly reflect changes in how filings are recorded in DOB NOW.
- Large ZIPs have more filings and more crashes; the lag analysis removes ZIP size, but the hotspot chart does not."""))

## 7. Research questions and visualizations (2 per member)
Write each question + 1 chart + 2-3 sentences of interpretation (no causal claims). The same function goes into your `charts_mX.py` file in the repo.

### M1: RQ1 and RQ2

**RQ1. Which ZIP codes had the largest growth in new-building applications between 2022 and 2025, and where are they?**
*Why it matters:* planners and transit agencies need to know where new housing activity is concentrating, so services can be prepared before demand arrives.
*Method:* we counted new-building initial applications per ZIP for 2022 and for 2025 (the last full year) and ranked ZIPs by the absolute increase. We ranked by absolute change instead of percentage because a ZIP going from 1 to 5 filings is +400% and would crowd out the larger shifts; the percentage is still shown on hover.
*Finding:* the 15 ZIPs with the largest increases (+9 to +23 applications) are all in the outer boroughs: 9 in Queens, 3 in Brooklyn and 3 in the Bronx, with none in Manhattan or Staten Island. The largest is 11691 in Queens (34 applications in 2022, 57 in 2025, +68%), and 7 of the 15 ZIPs at least doubled (for example 11106, from 4 to 13). Citywide, new-building initial applications were roughly flat in the raw download (2,483 in 2022, 2,386 in 2025), so these ZIPs show where activity shifted, not a citywide boom.
*Caution:* the counts are small (2022 base of 4 to 34), so a few projects can change the ranking. This counts applications, not buildings completed or people moving in. Counting every row, including follow-up filings, would show growth of over +130 per ZIP, but that mostly reflects amendments piling up on earlier projects. The ranking says nothing about causes.

**RQ2. How does the share of new-building applications (versus alterations) differ across boroughs and change over time?**
*Why it matters:* a borough dominated by alterations is changing mostly through renovation, while one with a higher new-building share is adding new structures. Each calls for different planning.
*Method:* for each borough and year (2022 to 2025) we computed new-building initial applications as a percentage of new-building plus alteration applications, using the team's shared job-type grouping (which counts "ALT-CO - New Building with Existing Elements to Remain" as a new building).
*Finding:* alterations dominate everywhere. Manhattan stays between 0.1% and 0.4%, Brooklyn between 2.5% and 3.9%, the Bronx between 2.2% and 4.8%, and Queens between 3.8% and 5.7%. Staten Island is the highest but swings widely: 16.3% (2022), 23.1% (2023), 7.7% (2024) and 8.4% (2025). In 2025 the share rose in every borough compared with 2024.
*Caution:* a share can rise because new-building applications grew or because alteration applications fell, so this chart cannot tell which. Staten Island has the fewest filings and loses about 20% of them in the ZIP step, so its share is the least reliable. The result depends on the grouping choice for the ALT-CO type, and it describes applications, not causes.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

FIRST_YEAR, LAST_FULL_YEAR = 2022, 2025   # 2026 is incomplete, so 2025 is the last full year
FILING_COLS = ['filings_new_building', 'filings_alteration', 'filings_other']

def make_zip_year_table(h):
    """Shape the cleaned table like merged_data.csv (one row per ZIP x year, filings by group),
    so the same chart functions also work on the website's data."""
    h = h.copy()
    h['borough'] = h.groupby('zip')['borough'].transform(lambda s: s.mode().iat[0])   # one borough per ZIP
    h['group'] = housing_group(h['job_type'])
    t = (h.groupby(['zip', 'borough', 'year', 'group']).size()
           .unstack('group', fill_value=0).add_prefix('filings_').reset_index())
    for c in FILING_COLS:
        if c not in t:
            t[c] = 0
    return t

viz_h = make_zip_year_table(df_housing)

# ---------------------------------------------------------------- RQ1
def rq1_growth(df, y0=FIRST_YEAR, y1=LAST_FULL_YEAR, top=15):
    """Which ZIPs gained the most new-building applications between y0 and y1?"""
    d = (df[df['year'].isin([y0, y1])]
         .groupby(['zip', 'borough', 'year'])['filings_new_building'].sum()
         .unstack('year', fill_value=0).reindex(columns=[y0, y1], fill_value=0).reset_index())
    d.columns = ['zip', 'borough', f'filings_{y0}', f'filings_{y1}']
    d['change'] = d[f'filings_{y1}'] - d[f'filings_{y0}']
    base = d[f'filings_{y0}'].where(d[f'filings_{y0}'] > 0)       # % change undefined when the base is 0
    d['pct_change'] = (d['change'] / base * 100).round(0)
    best = d.nlargest(top, 'change').sort_values('change')
    fig = px.bar(best, x='change', y='zip', color='borough', orientation='h',
                 hover_data={f'filings_{y0}': True, f'filings_{y1}': True, 'pct_change': True,
                             'change': True, 'borough': False},
                 labels={'change': f'Increase in new-building applications, {y0} to {y1}', 'zip': 'ZIP code'},
                 title=f'Top {top} ZIPs by growth in new-building applications ({y0} to {y1})')
    fig.update_yaxes(type='category', categoryorder='total ascending')
    return fig, best.sort_values('change', ascending=False)

# ---------------------------------------------------------------- RQ2
def rq2_share(df, y0=FIRST_YEAR, y1=LAST_FULL_YEAR):
    """Share of new-building applications among new-building + alteration applications, per borough and year."""
    d = (df[df['year'].between(y0, y1)].groupby(['borough', 'year'])[FILING_COLS].sum().reset_index())
    d['new_share'] = (d['filings_new_building'] / (d['filings_new_building'] + d['filings_alteration']) * 100).round(1)
    fig = px.line(d, x='year', y='new_share', color='borough', markers=True,
                  hover_data={'filings_new_building': True, 'filings_alteration': True},
                  labels={'new_share': 'New buildings as % of new + alteration applications', 'year': 'Year'},
                  title='Share of new-building applications by borough')
    fig.update_xaxes(dtick=1)
    return fig, d.pivot(index='borough', columns='year', values='new_share')

fig1, table1 = rq1_growth(viz_h); fig1.show(); print(table1.to_string(index=False))
fig2, table2 = rq2_share(viz_h); fig2.show(); print(table2)

### M2: RQ3 and RQ4
**RQ3. Which boroughs have the highest share of crashes with pedestrian or cyclist casualties, and has the share changed since 2022?**
*Why it matters:* pedestrians and cyclists are the most exposed road users, so a rising share shows safety teams where street design matters most.
*Method:* for each borough and year, crashes with at least one pedestrian injured or killed (and, separately, with at least one cyclist) as a percentage of all crashes. The two are shown as separate panels and not added, because one crash can be counted in both. 2026 covers January to October only and is labelled as such.

*Finding:* Manhattan consistently had the highest shares for both pedestrian- and cyclist-casualty crashes. The pedestrian share increased from 10.5% in 2022 to 12.3% in 2026, while the cyclist share rose from 8.9% to 10.1%. The Bronx also showed a notable increase in pedestrian-casualty crashes, from 8.4% to 10.6%.

*Caution:* 2026 covers only January–October, so it is not directly comparable to full-year results. These shares also do not account for differences in pedestrian or cyclist traffic between boroughs.

**RQ4. Which ZIPs have the most injuries per crash?**
*Why it matters:* ZIPs with many crashes are not necessarily where crashes hurt most; injuries per crash points to more severe crashes, which crash counts alone hide.
*Method:* total injuries divided by total crashes per ZIP over the whole window, shown per 100 crashes. The quarter of ZIPs with the fewest crashes is left out, because a ZIP with few crashes can top the list by chance. The top 15 are shown.

*Finding:* ZIP code 11236 in Brooklyn ranked highest, with 73.5 injuries per 100 crashes, followed by 10469 in the Bronx (72.5) and 11367 in Queens (71.4). All 15 ZIP codes in the ranking exceeded the citywide average of 56.0 injuries per 100 crashes.

*Caution:* ZIP codes with fewer than 608 crashes were excluded (54 out of 205 ZIPs). Injuries per 100 crashes measures the number of injured people, not the percentage of crashes involving injuries, and does not account for differences in traffic volume or road-user exposure.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

def _empty(message):
    return go.Figure().add_annotation(text=message, showarrow=False)


# ---------------------------------------------------------------- RQ3
def rq3(df):
    """Share of crashes with a pedestrian / cyclist casualty, by borough and year."""
    d = (df.groupby(["borough", "year"], as_index=False)
           [["crashes_total", "crashes_pedestrian", "crashes_cyclist"]].sum())
    d = d[d["crashes_total"] > 0]
    if d.empty:
        return _empty("No crashes for this filter")
    last_month = df.groupby("year")["month"].max()
    d["period"] = [str(y) if last_month[y] == 12 else f"{y} (to month {last_month[y]})" for y in d["year"]]
    d["Pedestrian"] = d["crashes_pedestrian"] / d["crashes_total"] * 100
    d["Cyclist"] = d["crashes_cyclist"] / d["crashes_total"] * 100
    long = d.melt(id_vars=["borough", "period", "crashes_total"], value_vars=["Pedestrian", "Cyclist"],
                  var_name="Casualty", value_name="share")
    fig = px.line(long, x="period", y="share", color="borough", facet_col="Casualty", markers=True,
                  hover_data={"crashes_total": True, "share": ":.1f"},
                  labels={"share": "% of crashes", "period": "Year", "borough": "Borough"})
    fig.update_yaxes(matches=None, showticklabels=True)
    fig.update_xaxes(categoryorder="category ascending")
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1] + " casualty"))
    return fig


# ---------------------------------------------------------------- RQ4
def rq4(df, top=15):
    """ZIPs with the most injuries per 100 crashes (the quarter of ZIPs with the fewest crashes is left out)."""
    d = df.groupby(["zip", "borough"], as_index=False)[["crashes_total", "injuries", "fatalities"]].sum()
    d = d[d["crashes_total"] > 0]
    if d.empty:
        return _empty("No crashes for this filter")
    d = d[d["crashes_total"] >= d["crashes_total"].quantile(0.25)]
    d["injuries_per_100"] = (d["injuries"] / d["crashes_total"] * 100).round(1)
    best = d.nlargest(top, "injuries_per_100").sort_values("injuries_per_100")
    fig = px.bar(best, x="injuries_per_100", y="zip", color="borough", orientation="h",
                 hover_data={"crashes_total": True, "injuries": True, "fatalities": True, "borough": False},
                 labels={"injuries_per_100": "Injuries per 100 crashes", "zip": "ZIP code", "borough": "Borough"})
    fig.update_yaxes(type="category", categoryorder="total ascending")
    return fig

In [ ]:
# ---- RQ3
fig3 = rq3(final)
fig3.show()
t3 = final.groupby(['borough', 'year'])[['crashes_total', 'crashes_pedestrian', 'crashes_cyclist']].sum()
print('Pedestrian-casualty crashes, % of all crashes:')
print((t3['crashes_pedestrian'] / t3['crashes_total'] * 100).round(1).unstack('year'))
print('\nCyclist-casualty crashes, % of all crashes:')
print((t3['crashes_cyclist'] / t3['crashes_total'] * 100).round(1).unstack('year'))

# ---- RQ4
fig4 = rq4(final); fig4.show()
z = final.groupby(['zip', 'borough'])[['crashes_total', 'injuries', 'fatalities']].sum().reset_index()
cut = z['crashes_total'].quantile(0.25)
print(f"\nZIPs: {len(z)} | cut-off (25th percentile of crashes): {cut:,.0f} | ZIPs left out: {(z['crashes_total'] < cut).sum()}")
z = z[z['crashes_total'] >= cut].copy()
z['injuries_per_100'] = (z['injuries'] / z['crashes_total'] * 100).round(1)
print(z.nlargest(15, 'injuries_per_100').to_string(index=False))
print('Citywide injuries per 100 crashes:', round(final['injuries'].sum() / final['crashes_total'].sum() * 100, 1))

### M3: RQ5 and RQ6

**RQ5. Which ZIP codes rank in the top quarter for BOTH new-building filings and crashes?**
Why it matters: these are the places where new construction and street danger coincide, so they are where planners and safety teams should look first.
Method: total new-building filings and total crashes per ZIP over the study window, converted to percentile ranks among ZIPs; hotspot = top quarter on both.
*Findings:* 24 of 205 ZIPs (12%) rank in the top quarter on both new-building filings and crashes, compared with about 6% expected if the two were unrelated. Across all ZIPs the rank correlation is 0.69, a moderate positive relationship. The hotspots are in Brooklyn (12 ZIPs), Queens (9) and the Bronx (3), with none in Manhattan or Staten Island.
*Caution:* busier, larger ZIPs have more of everything, so this shows WHERE the two overlap, not that construction causes crashes.

**RQ6. Within a ZIP, do months with unusually many filings come before months with unusually many crashes?**
Why it matters: if construction raised crash risk, unusually busy months should be followed by unusually bad months.
Method: for each ZIP and month, subtract the ZIP's own average and the citywide average of that month (this removes ZIP size and seasonality), then correlate filings in month t with crashes in month t+lag, for lags 0 to 6.
*Findings:* After removing each ZIP's own average and each month's citywide average, the correlation between filings in one month and crashes in the following months is between -0.017 and 0.013 for every lag from 0 to 6 months. Values this small mean unusually busy filing months were not followed by noticeably worse crash months.
*Caution:* correlation is not causation. Filings are paperwork, not construction started, and crash reporting depends on police recording.

**Taken together:** ZIPs with more new-building filings tend to also have more crashes (RQ5), but within a ZIP the timing shows almost no link (RQ6). This suggests the overlap reflects which neighbourhoods are dense and busy, not that construction timing drives crashes. We cannot establish causation with this data.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

HOT_Q = 0.75      # "high" = top quarter of ZIP codes
MAX_LAG = 6       # months


# ---------------------------------------------------------------- RQ1
def zip_summary(df):
    """One row per ZIP: totals, percentile rank in each domain, and a both-high flag."""
    d = df.groupby(["zip", "borough"], as_index=False)[["housing_metric", "transport_metric"]].sum()
    d["housing_pct"] = d["housing_metric"].rank(pct=True)
    d["transport_pct"] = d["transport_metric"].rank(pct=True)
    d["hotspot"] = (d["housing_pct"] >= HOT_Q) & (d["transport_pct"] >= HOT_Q)
    return d


def rq1(df):
    d = zip_summary(df)
    if d.empty:
        return go.Figure()
    d["marker_size"] = np.where(d["hotspot"], 15, 8)
    fig = px.scatter(d, x="housing_pct", y="transport_pct", color="borough", size="marker_size",
                     size_max=15, hover_name="zip",
                     hover_data={"housing_metric": True, "transport_metric": True,
                                 "housing_pct": ":.0%", "transport_pct": ":.0%", "marker_size": False},
                     labels={"housing_pct": "Construction activity (percentile among ZIPs)",
                             "transport_pct": "Crashes (percentile among ZIPs)",
                             "housing_metric": "Filings", "transport_metric": "Crashes"})
    # shade the "high on both" corner
    fig.add_shape(type="rect", x0=HOT_Q, x1=1.02, y0=HOT_Q, y1=1.02, fillcolor="rgba(200,68,47,0.10)",
                  line=dict(width=0), layer="below")
    n_hot, n = int(d["hotspot"].sum()), len(d)
    fig.add_annotation(x=1.0, y=1.0, xanchor="right", yanchor="bottom", showarrow=False,
                       text=f"{n_hot} of {n} ZIPs are high on both", font=dict(size=12, color="#c8442f"))
    fig.update_xaxes(tickformat=".0%", range=[0, 1.05])
    fig.update_yaxes(tickformat=".0%", range=[0, 1.05])
    return fig


# ---------------------------------------------------------------- RQ2
def _adjusted_panel(df, col):
    """ZIP x month matrix with each ZIP's own average and each month's citywide average removed.
    What is left = 'more or less than usual for this ZIP, in this month'."""
    wide = df.pivot_table(index="zip", columns="year_month", values=col, aggfunc="sum")
    months = pd.period_range(wide.columns.min(), wide.columns.max(), freq="M").astype(str)
    wide = wide.reindex(columns=months)               # missing months stay NaN, so lags never skip a gap
    w = wide.sub(wide.mean(axis=1), axis=0)
    w = w.sub(w.mean(axis=0), axis=1)
    return w.to_numpy()


def lag_correlations(df, max_lag=MAX_LAG):
    """Correlation between filings in month t and crashes in month t+lag, pooled over ZIPs."""
    if df.empty:
        return pd.DataFrame(columns=["lag", "r", "n"])
    h, t = _adjusted_panel(df, "housing_metric"), _adjusted_panel(df, "transport_metric")
    n_months, rows = h.shape[1], []
    for k in range(max_lag + 1):
        if n_months - k < 3:
            break
        a, b = h[:, : n_months - k], t[:, k:]
        ok = np.isfinite(a) & np.isfinite(b)
        r = np.nan
        if ok.sum() > 10:
            with np.errstate(invalid="ignore"):
                r = np.corrcoef(a[ok], b[ok])[0, 1]
        rows.append({"lag": k, "r": r, "n": int(ok.sum())})
    return pd.DataFrame(rows)


def rq2(df):
    lc = lag_correlations(df)
    if lc.empty or lc["r"].isna().all():
        return go.Figure().add_annotation(text="Not enough months for this filter", showarrow=False)
    fig = go.Figure(go.Bar(
        x=lc["lag"], y=lc["r"], text=lc["r"].round(3), textposition="outside",
        marker_color=np.where(lc["r"] >= 0, "#1f6f8b", "#c8442f"),
        customdata=lc["n"], hovertemplate="Lag %{x} months<br>r = %{y:.3f}<br>%{customdata:,} ZIP-month pairs<extra></extra>"))
    fig.update_layout(xaxis_title="Months between filings and crashes (0 = same month)",
                      yaxis_title="Correlation (ZIP- and month-adjusted)", xaxis=dict(dtick=1))
    fig.add_hline(y=0, line_width=1, line_color="#17212b")
    return fig

In [ ]:
# Metric used in the notebook version of M3's questions: NEW-BUILDING filings vs ALL crashes
viz = final.assign(housing_metric=final['filings_new_building'], transport_metric=final['crashes_total'])

def summarize_rq1(d):
    hot = d[d['hotspot']].sort_values('transport_metric', ascending=False)
    n, n_hot = len(d), len(hot)
    rank_corr = d['housing_pct'].corr(d['transport_pct'])          # Pearson on ranks = Spearman
    print(f'{n_hot} of {n} ZIPs ({n_hot / n:.0%}) are in the top quarter on BOTH. If the two were unrelated we would expect ~6% (0.25 x 0.25).')
    print(f'Rank correlation between construction and crashes across ZIPs: {rank_corr:.2f}')
    print('Hotspots by borough:'); print(hot['borough'].value_counts().to_string())
    print('\nTop hotspot ZIPs:'); print(hot[['zip', 'borough', 'housing_metric', 'transport_metric']].head(10).to_string(index=False))

def summarize_rq2(lc):
    best = lc.loc[lc['r'].abs().idxmax()]
    print(lc.round(3).to_string(index=False))
    print(f"\nStrongest association: lag {int(best['lag'])} month(s), r = {best['r']:.3f} on {int(best['n']):,} ZIP-month pairs.")
    print('Rule of thumb: |r| < 0.1 is very weak. Report weak or zero results honestly; that is a finding too.')

In [ ]:
# ---- RQ5
d_rq1 = zip_summary(viz)
rq1(viz).show()
summarize_rq1(d_rq1)

# ---- RQ6
lc = lag_correlations(viz)
rq2(viz).show()
summarize_rq2(lc)

### **M4: RQ7 and RQ8**
### **RQ7: Do new-building filings and crashes follow the same rhythm through the year?**
*Why it matters:* if filings and crashes peak in the same months, one calendar could serve both planners and safety teams. If they do not, the busy months for construction paperwork are not the busy months on the street.

***Method:*** for each borough and calendar month we averaged the monthly totals over the years in the data and divided by that borough's average month, so 100 is a typical month and 129 is 29% busier. One heatmap shows new-building filings and one shows all crashes, and we correlated the two 12-month profiles.

**Finding:** the two yearly shapes are only weakly alike. Citywide the correlation between the profiles is 0.26. By borough it is close to zero in the Bronx (0.00) and Brooklyn (0.06), small in Manhattan (0.16) and Queens (0.25), and highest in Staten Island (0.57). Crashes follow a smooth curve: lowest in February in every borough (index 82 to 90) and highest in May in four boroughs (107 to 111), with Staten Island peaking in December. Filings are lumpier. Citywide they sit at 83 to 89 in January and February, jump to 129 in March, stay between 94 and 100 from April to November, and rise again to 123 in December. March is the filings peak in four boroughs (Manhattan 186, Bronx 135, Brooklyn 129, Queens 128), while Staten Island peaks in December (205).
*Caution:* each profile has only 12 points, so one unusual month moves the correlation a lot. The March and December filing peaks may reflect administrative timing such as year-end or deadline rushes, and the housing EDA (section 3) found an unexplained spike in early 2026 that could lift March; we cannot confirm either from this data. Manhattan and Staten Island have few new-building filings, so their indexes (Staten Island 205, Manhattan 47 in July) rest on small counts. 2026 is incomplete, so November and December average one year fewer than the other months. This describes the calendar only and says nothing about cause.

### **RQ8: Where do heavy new-building activity and a high share of pedestrian-casualty crashes overlap?**
*Why it matters:* RQ5 shows where construction and crashes overlap in volume, but large ZIPs have more of everything. The pedestrian-casualty share asks something different: in which ZIPs is a crash more likely to hurt someone on foot, and are those the ZIPs with heavy construction?

***Method:*** for each ZIP we summed new-building filings (bubble size) and computed crashes with at least one pedestrian injured or killed as a percentage of all crashes (colour). The 31 ZIPs with fewer than 100 crashes are left out, because a share built on so few crashes is mostly noise. "High on both" means the top quarter of the 174 ZIPs shown, on filings and on pedestrian share.

***Finding:*** the overlap is small. Only 9 of 174 ZIPs are high on both, fewer than the 11 expected if the two were unrelated, and across ZIPs the rank correlation is weakly negative (-0.21): ZIPs with more new-building filings tend to have a slightly lower pedestrian share. The 9 are in Brooklyn (5) and Queens (4), with none in Manhattan, the Bronx or Staten Island. The highest shares among them are 11219 in Brooklyn (16.1%), 11355 in Queens (15.7%), 11204 in Brooklyn (14.9%) and 11230 in Brooklyn (14.8%). Across the ZIPs shown, 9.8% of crashes involve an injured or killed pedestrian. By borough the share is highest in Manhattan (11.4%), then the Bronx (10.1%), Brooklyn (9.9%), Queens (8.9%) and Staten Island (7.8%). Manhattan has the highest share but very few new-building filings (RQ2), so the two rarely coincide there.
*Caution:* the share is not a rate per person walking, and the data has no pedestrian counts, so busy walking areas will score high whatever the construction level. A ZIP with many car crashes will also have a lower pedestrian share. Many ZIPs have only a handful of new-building filings, and 9 versus 11 is too small a difference to read as meaningfully below chance. ZIP codes recovered from coordinates can be off near boundaries. This describes association only.

**Taken together (RQ7 and RQ8):** neither the yearly calendar nor the pedestrian share shows a strong link between construction and crashes, which fits the within-ZIP result in RQ6. The overlap in RQ5 looks like a matter of ZIP size and density more than of construction itself. This data cannot establish causation.

In [ ]:
%%writefile charts_m4.py
"""MEMBER 4 - cross-domain questions: seasonality and the map overlay.

RQ7  Do construction filings and crashes follow the same rhythm through the year?
     (borough x calendar-month heatmaps of a seasonal index; 100 = an average month)
RQ8  Where do heavy construction activity and a high share of pedestrian-casualty crashes overlap?
     (ZIP-centroid map: bubble size = filings, colour = pedestrian-casualty crashes as % of all crashes)

Every chart is a function f(df) -> plotly Figure. `df` already has the `housing_metric` and
`transport_metric` columns added by data_layer.apply_filters. The summarize_* helpers return
the exact numbers behind each chart so the notebook text can quote them.
"""
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
BORO_ORDER = ["BRONX", "BROOKLYN", "MANHATTAN", "QUEENS", "STATEN ISLAND"]
MIN_CRASHES = 100          # RQ8: a share built on fewer crashes is mostly noise
# diverging scale shared with M3's palette: below average = teal, above average = red
DIVERGING = [[0, "#1f6f8b"], [0.5, "#f7f7f7"], [1, "#c8442f"]]


def _empty(message):
    fig = go.Figure()
    fig.add_annotation(text=message, showarrow=False, font=dict(size=14))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig


# ---------------------------------------------------------------- RQ7
def seasonal_index(df, col):
    """Borough x calendar-month table. Each cell = the average monthly total for that calendar month
    (averaged over the years in the data), divided by the borough's average across all 12 months, x100.
    100 = a typical month; 130 = 30% busier than a typical month for that borough."""
    monthly = df.groupby(["borough", "year", "month"], as_index=False)[col].sum()
    avg = monthly.groupby(["borough", "month"])[col].mean().unstack("month").reindex(columns=range(1, 13))
    index = avg.div(avg.mean(axis=1).replace(0, np.nan), axis=0) * 100
    order = [b for b in BORO_ORDER if b in index.index]
    return index.loc[order]


def rq7(df):
    if df.empty:
        return _empty("No data for this filter")
    housing, transport = seasonal_index(df, "housing_metric"), seasonal_index(df, "transport_metric")
    if housing.empty or transport.empty:
        return _empty("No data for this filter")
    span = float(np.nanmax(np.abs(np.concatenate([housing.to_numpy().ravel(), transport.to_numpy().ravel()]) - 100)))
    span = max(span, 10.0)

    fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.03,
                        subplot_titles=("Construction filings", "Crashes"))
    for col, (name, table) in enumerate([("Filings", housing), ("Crashes", transport)], start=1):
        fig.add_trace(go.Heatmap(
            z=table.to_numpy(), x=MONTHS, y=table.index.tolist(), coloraxis="coloraxis",
            texttemplate="%{z:.0f}", textfont=dict(size=10),
            hovertemplate=f"{name}<br>%{{y}}, %{{x}}<br>Seasonal index %{{z:.0f}} (100 = average month)<extra></extra>"),
            row=1, col=col)
    fig.update_layout(
        coloraxis=dict(colorscale=DIVERGING, cmid=100, cmin=100 - span, cmax=100 + span,
                       colorbar=dict(title="Index<br>(100 = avg month)", thickness=12)),
        margin=dict(l=10, r=10, t=40, b=10))
    fig.update_yaxes(autorange="reversed")
    return fig


def summarize_rq7(df):
    """Numbers for the notebook: how alike are the two seasonal profiles, and when does each peak?"""
    housing, transport = seasonal_index(df, "housing_metric"), seasonal_index(df, "transport_metric")
    rows = []
    for b in housing.index:
        h, t = housing.loc[b], transport.loc[b]
        r = h.corr(t) if h.notna().sum() >= 6 else np.nan
        rows.append(dict(borough=b, profile_correlation=round(r, 2),
                         filings_peak=MONTHS[int(h.idxmax()) - 1], filings_peak_index=round(h.max()),
                         filings_low=MONTHS[int(h.idxmin()) - 1], filings_low_index=round(h.min()),
                         crashes_peak=MONTHS[int(t.idxmax()) - 1], crashes_peak_index=round(t.max()),
                         crashes_low=MONTHS[int(t.idxmin()) - 1], crashes_low_index=round(t.min())))
    table = pd.DataFrame(rows)
    city = df.groupby(["year", "month"], as_index=False)[["housing_metric", "transport_metric"]].sum()
    city = city.groupby("month")[["housing_metric", "transport_metric"]].mean().reindex(range(1, 13))
    city_r = city["housing_metric"].corr(city["transport_metric"])
    print(table.to_string(index=False))
    print(f"\nCitywide: correlation between the 12-month filings profile and the 12-month crash profile = {city_r:.2f}")
    print("Citywide seasonal index (100 = average month):")
    idx = (city / city.mean() * 100).round(0)
    idx.index = MONTHS
    print(idx.T.to_string())
    return table


# ---------------------------------------------------------------- RQ8
def zip_map_table(df, min_crashes=MIN_CRASHES):
    """One row per ZIP: filings, crashes, pedestrian-casualty share. ZIPs with fewer than min_crashes
    crashes are excluded from the colour scale because their share is mostly noise."""
    d = df.groupby(["zip", "borough"], as_index=False).agg(
        lat=("lat", "mean"), lon=("lon", "mean"),
        housing_metric=("housing_metric", "sum"), crashes_total=("crashes_total", "sum"),
        crashes_pedestrian=("crashes_pedestrian", "sum"))
    d = d[d["crashes_total"] > 0].copy()
    d["ped_share"] = d["crashes_pedestrian"] / d["crashes_total"] * 100
    n_excluded = int((d["crashes_total"] < min_crashes).sum())
    kept = d[d["crashes_total"] >= min_crashes]
    if len(kept) < 5:                       # a very narrow filter: show everything rather than an empty map
        kept, n_excluded = d, 0
    return kept.copy(), n_excluded


def rq8(df):
    if df.empty:
        return _empty("No data for this filter")
    d, n_excluded = zip_map_table(df)
    if d.empty:
        return _empty("No crashes for this filter")
    d["bubble"] = d["housing_metric"].clip(lower=0.5)         # zero would make the bubble vanish
    fig = px.scatter_map(
        d, lat="lat", lon="lon", size="bubble", size_max=28, color="ped_share",
        color_continuous_scale="YlOrRd", hover_name="zip",
        hover_data={"borough": True, "housing_metric": True, "crashes_total": True,
                    "ped_share": ":.1f", "bubble": False, "lat": False, "lon": False},
        labels={"ped_share": "Pedestrian-casualty<br>crashes (% of all)", "housing_metric": "Filings",
                "crashes_total": "Crashes", "borough": "Borough"},
        center=dict(lat=40.71, lon=-73.95), zoom=9.2, map_style="carto-positron")
    fig.update_traces(marker=dict(opacity=0.75))
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0), coloraxis_colorbar=dict(thickness=12))
    if n_excluded:
        fig.add_annotation(xref="paper", yref="paper", x=0, y=0, xanchor="left", yanchor="bottom",
                           showarrow=False, bgcolor="rgba(255,255,255,0.8)", font=dict(size=11),
                           text=f"{n_excluded} ZIPs with fewer than {MIN_CRASHES} crashes are not shown")
    return fig


def summarize_rq8(df):
    """Numbers for the notebook: which ZIPs combine heavy filings with a high pedestrian share?"""
    d, n_excluded = zip_map_table(df)
    d["housing_pct"] = d["housing_metric"].rank(pct=True)
    d["ped_pct"] = d["ped_share"].rank(pct=True)
    both = d[(d["housing_pct"] >= 0.75) & (d["ped_pct"] >= 0.75)].sort_values("ped_share", ascending=False)
    print(f"ZIPs shown: {len(d)} (excluded for fewer than {MIN_CRASHES} crashes: {n_excluded})")
    print(f"Citywide pedestrian-casualty share among shown ZIPs: "
          f"{d['crashes_pedestrian'].sum() / d['crashes_total'].sum() * 100:.1f}%")
    print(f"Rank correlation, filings vs pedestrian share across ZIPs: {d['housing_pct'].corr(d['ped_pct']):.2f}")
    print(f"ZIPs in the top quarter on BOTH filings and pedestrian share: {len(both)} of {len(d)} "
          f"(about {0.25 * 0.25 * 100:.0f}% = {0.0625 * len(d):.0f} expected if unrelated)")
    print("\nBoroughs of those ZIPs:"); print(both["borough"].value_counts().to_string())
    print("\nTop ZIPs:")
    print(both[["zip", "borough", "housing_metric", "crashes_total", "ped_share"]].round(1).head(8).to_string(index=False))
    print("\nPedestrian share by borough (all shown ZIPs):")
    g = d.groupby("borough")[["crashes_pedestrian", "crashes_total"]].sum()
    print((g["crashes_pedestrian"] / g["crashes_total"] * 100).round(1).to_string())
    return d


CHARTS = [
    ("M4-RQ7: Do construction filings and crashes follow the same rhythm through the year?", rq7),
    ("M4-RQ8: Where do heavy construction activity and a high pedestrian-casualty share overlap?", rq8),
]

In [ ]:
import importlib
import charts_m4 as m4          # written by the cell above, no upload needed
importlib.reload(m4)

# same metrics as Member 3's cells: new-building filings vs all crashes
viz = final.assign(housing_metric=final['filings_new_building'], transport_metric=final['crashes_total'])

# ---- RQ7: seasonal rhythm
m4.rq7(viz).show()
seasonality = m4.summarize_rq7(viz)

# ---- RQ8: ZIP map
m4.rq8(viz).show()
zip_map = m4.summarize_rq8(viz)

## 8. Story: question, discovery, limits, impact

The text below is generated from the final data in `final`, so it matches the website's story section.

In [ ]:
# Builds the story text from the data, so it always matches the website
from IPython.display import Markdown, display
import importlib
import numpy as np
import pandas as pd

# zip_summary and lag_correlations come from Member 3's cells in section 7; m4 from the cell above

MONTH_NAMES = ["January", "February", "March", "April", "May", "June", "July", "August",
               "September", "October", "November", "December"]

def describe_r(r):
    a, sign = abs(r), ("positive" if r >= 0 else "negative")
    if a < 0.1:
        return "almost no relationship"
    return f"a {'weak' if a < 0.3 else 'moderate' if a < 0.7 else 'strong'} {sign} relationship"

def borough_list(counts):
    parts = [f"{b.title()} ({n})" for b, n in counts.items() if n > 0]
    return ", ".join(parts[:-1]) + f" and {parts[-1]}" if len(parts) > 1 else "".join(parts)

# same metrics as the website story: new-building filings vs all crashes
base = final.assign(housing_metric=final['filings_new_building'], transport_metric=final['crashes_total'])
z = zip_summary(base)
lag = lag_correlations(base).dropna(subset=['r'])
city = (base.groupby(['year', 'month'])[['housing_metric', 'transport_metric']].sum()
            .groupby('month').mean().reindex(range(1, 13)))
map_table, _ = m4.zip_map_table(base)

first, last = pd.Period(final['year_month'].min(), freq='M'), pd.Period(final['year_month'].max(), freq='M')
window = f"{MONTH_NAMES[first.month - 1]} {first.year} to {MONTH_NAMES[last.month - 1]} {last.year}"
n_zips = final['zip'].nunique()
n_hot, n = int(z['hotspot'].sum()), len(z)
rank_r = z['housing_pct'].corr(z['transport_pct'])
hot_boroughs = z[z['hotspot']]['borough'].value_counts().to_dict()
lag_lo, lag_hi, lag_abs, lag_last = lag['r'].min(), lag['r'].max(), lag['r'].abs().max(), int(lag['lag'].max())
season_r = city['housing_metric'].corr(city['transport_metric'])
crash_peak = MONTH_NAMES[int(city['transport_metric'].idxmax()) - 1]
crash_low = MONTH_NAMES[int(city['transport_metric'].idxmin()) - 1]
filing_peak = MONTH_NAMES[int(city['housing_metric'].idxmax()) - 1]
ped_r = map_table['housing_metric'].rank(pct=True).corr(map_table['ped_share'].rank(pct=True))
zero_share = (final['filings_new_building'] == 0).mean()
lag_weak = lag_abs < 0.1

story = f"""### The question

Do the parts of New York with the most new construction also have more street-safety problems? We lined up Department of Buildings job filings with police-recorded motor vehicle crashes, ZIP code by ZIP code and month by month, across {n_zips} ZIP codes from {window}.

It matters because the people who decide where housing goes and the people who decide where to fix streets rarely work from the same map.

### What we found

**Across ZIP codes.** ZIP codes with more new-building filings also tend to have more crashes ({describe_r(rank_r)}, rank correlation {rank_r:.2f}). {n_hot} of {n} ZIP codes are in the top quarter on both, where about {0.0625 * n:.0f} would be expected if the two were unrelated{". They are in " + borough_list(hot_boroughs) + "." if hot_boroughs else "."}

**Within one ZIP code.** After removing its usual level and each month's citywide pattern, filings and crashes up to {lag_last} months later correlate between {lag_lo:.3f} and {lag_hi:.3f}. That is {describe_r(lag_abs)}{": unusually busy filing months were not followed by noticeably worse crash months." if lag_weak else ": busier filing months were followed by somewhat busier crash months, which on its own does not show cause."}

**Through the year.** Crashes are highest in {crash_peak} and lowest in {crash_low}, while new-building filings peak in {filing_peak}. Across the year the two patterns have {describe_r(season_r)} (r = {season_r:.2f}).

**Pedestrians.** Across ZIP codes, the share of crashes that injure or kill a pedestrian has {describe_r(ped_r)} to the number of new-building filings (rank correlation {ped_r:.2f}).

{"Taken together, the overlap looks like geography: dense, busy neighbourhoods have more of both. It does not look like construction timing driving crashes." if rank_r >= 0.3 and lag_weak else "These are patterns of association only."}

### What the data cannot tell us

Correlation is not causation. Nothing here shows that construction causes crashes, or that crashes shape where people build.

A filing is an application, not a finished building or a household moving in, and its date can sit months away from the work.

The data has no traffic counts, so busy ZIP codes naturally have more crashes. Crash records also depend on police reporting.

Many filings and some crashes had no usable ZIP code, so we placed them using their coordinates. Records near a ZIP boundary can land in the neighbouring ZIP. About {zero_share:.0%} of ZIP-months have no new-building filings at all, so small counts can swing results.

### Why it matters for New York

Planners can start safety reviews in the ZIP codes where heavy construction and frequent crashes already coincide, instead of waiting for new housing to open.

{"Because busy filing months were not followed by worse crash months, a spike in filings is not an early warning. Safety teams will get more from tracking persistent hotspots." if lag_weak else "Busier filing months were followed by somewhat busier crash months, so safety teams could watch filing spikes as one signal, alongside persistent hotspots."}

Housing and street-safety advocates can use the map and filters on our website to check their own neighbourhood. The next step is to add traffic and population counts so the comparison is per person, not per ZIP code.
"""
display(Markdown(story))

## 9. Prompt log (everyone)

| Member | Step | Prompt used |
|---|---|---|
| Member | Step | Prompt used |
| M1 | Housing EDA and cleaning | "Explain the steps for performing EDA and cleaning on the housing dataset, providing the code step by step." |
| M1 | ZIP diagnosis | "Check whether the housing ZIP column can be trusted as a geographic join key." |
| M1 | Cleaning decisions | "Write Markdown explanations justifying each cleaning decision made for the housing dataset, based on the EDA outputs." |
| M1 | Research questions | "Develop RQ1 and RQ2 (ZIP growth in new-building filings and new-building share by borough) and suggest appropriate visualizations." |
| M2 | Transport EDA and cleaning | "Explain the steps needed to perform EDA and clean the transport dataset, providing code and guidance for each step." |
| M2 | ZIP diagnosis | "Interpret the ZIP code cleaning and recovery outputs for the transport dataset and explain what steps should be taken next." |
| M2 | Cleaning decisions | "Write Markdown explanations for the transport dataset's EDA findings and cleaning decisions based on the actual outputs." |
| M2 | Research questions| "Help develop RQ3 and RQ4, first is pedestrian and cyclist crash shares by borough over time and second is injuries per crash by ZIP." |
| M3 | Project planning | "Explain what needs to be done for the project and distribute the tasks among four members." |
| M3 | Download fix | "Colab gives a ConnectionResetError while downloading the housing dataset. Help troubleshoot and fix it." |
| M4 | Role and deliverables | "we divided the work over 4 members and im member 4 who: owns the look of the site (app.py and style.css), writes the story section, deploys on render, and handles README, the github repo and the submission email. charts go in charts_m4.py." (Attached: project description PDF, the team notebook.) |
| M4 | Dataset and integration decision | "we picked housing: H5 DOB NOW Job Filings, transportation: T4 crashes, Integration: ZIP x month, Question: do areas with heavy new construction also have more street safety problems? tell me where to start keep in mind im using google colab that uses cells" (Attached: dataset catalog, data_layer.py, charts_m1.py to charts_m3.py.) |
| M4 | Notebook debugging | "i uploaded charts_m4.py into the content in the notebook how should i change this accordingly" followed by the Colab error output and a screenshot of the Files panel. Used to fix the import path. |

## 10. Contributions

**Member 1 (housing data):** EDA of the DOB NOW job filings (job types, boroughs, filings over time, missing values); pre-integration cleaning (date parsing, borough standardisation, duplicate removal by filing number, domain-rule and IQR outlier handling, missing-value decisions); found that the published ZIP column does not match the building location and derived ZIP from coordinates instead; delivered `df_housing` to the cleaning contract; research questions RQ1 (ZIP growth in new-building filings) and RQ2 (new-building share by borough) with charts in `charts_m1.py`.

**Member 2 (transport data):** followed a similar EDA and pre-integration cleaning process to Member 1's housing dataset, but applied it to the NYC motor vehicle collisions dataset; investigated crash distributions, missing values, borough and ZIP code inconsistencies, and injury/fatality outliers; standardised dates and borough labels, removed duplicates, recovered missing ZIP codes using crash coordinates where possible, and documented cleaning decisions based on the dataset's characteristics; delivered the cleaned `df_transport` and `df_transport_full` datasets to the shared cleaning contract; developed research questions RQ3 (pedestrian and cyclist crash shares by borough over time) and RQ4 (injuries per crash by ZIP), with charts in `charts_m2.py`.

**Member 3 (integration and data layer):** designed the shared schema (`data_layer.py`) and the cleaning contract for M1/M2; wrote the robust downloader, ZIP-recovery helpers, ZIP x month aggregation, join, post-integration cleaning and validation; implemented the filter logic and the search-text parser for the website; built the merged dataset (`data/merged_data.csv`); research questions RQ5 (hotspot ZIPs in both domains) and RQ6 (lag relationship on ZIP- and month-adjusted data); reviewed teammates' cleaned tables for contract compliance and merged branches.

**Member 4 (website and story)**: designed and built the interactive website (app.py, assets/style.css) with Dash and Plotly: borough, year, housing job type and crash type dropdowns, a search mode that turns typed text into filters, and a Generate report button that redraws every chart; combined the charts of all four members into one page with a shared look; wrote the story section (question, findings, limits, impact), with its numbers computed from the data; research questions RQ7 (seasonal rhythm of filings and crashes by borough) and RQ8 (ZIP map of filings against pedestrian-casualty share), charts in charts_m4.py; deployed the site on Render; wrote the README; set up the GitHub repository; sent the submission email.